In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import gc
import pickle

import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt

from datetime import datetime
from pandas.tseries.offsets import MonthEnd

In [2]:
os.listdir('/data/aman_singh/acuuracy_check')

['Heuristics_all_combination_qcom_cp_apr_live.xlsx',
 'chek_nan.csv',
 'Heuristics_all_combination_ecom_mar_live.xlsx',
 'combine_model+missing_forecasts_brand_asm.ipynb',
 'MARICO LIMITED_swiggy_june.xlsx',
 'April-26 Plans.xlsx',
 'QCOM Chain PSKU OTP Output',
 'Heuristics_all_combination_qcom_chain_psku_june_live.xlsx',
 'Norms 202606.csv',
 'key_check.csv',
 'Norms 202607.csv',
 'all_combination_qcom_july_pred.csv',
 'Stat Demand Forecast MT_as_on_11th_May_2026.xlsb',
 'acc_framework_may_final.xlsx',
 'acc_offtakes_till_may.csv',
 'swigy_vol_chk.csv',
 'ALL Channels Accuracy_fva.ipynb',
 'Marico Ltd._forecast_Jul 2026_to_Oct 2026.csv',
 'all_combination_ecom_backtest_pred.csv',
 'trend_file_train_till_31_Jul_2026 (2).csv',
 'ecom_chain_psku_offtake_to_secondary_v6_PROD.ipynb',
 'seasonality.xlsx',
 'Qcom_chain_fc_psku_forecast_as_on_11th_may_2026.xlsx',
 'missing_df_gt_all.csv',
 'SOH - 01 Jul.xlsx',
 'SOH - 01 Jun.xlsx',
 'qcom_chain_depot_psku_zepto_inc.xlsx',
 'stat_fva_till_may

In [3]:
base_dir = '/data/aman_singh/acuuracy_check'
input_table = 'TRN_DF_QCOM_OFFTAKE_CHAIN_DEPOT_PSKU'
run_month = '2026-08-31'

In [4]:
def list_all_files_in_directory(root):
    out = []

    for path, subdirs, files in os.walk(root):
        for name in files:
            out.append(os.path.join(path, name))

    return out

In [5]:
list_all_files_in_directory(base_dir)

['/data/aman_singh/acuuracy_check/Heuristics_all_combination_qcom_cp_apr_live.xlsx',
 '/data/aman_singh/acuuracy_check/chek_nan.csv',
 '/data/aman_singh/acuuracy_check/Heuristics_all_combination_ecom_mar_live.xlsx',
 '/data/aman_singh/acuuracy_check/combine_model+missing_forecasts_brand_asm.ipynb',
 '/data/aman_singh/acuuracy_check/MARICO LIMITED_swiggy_june.xlsx',
 '/data/aman_singh/acuuracy_check/April-26 Plans.xlsx',
 '/data/aman_singh/acuuracy_check/Heuristics_all_combination_qcom_chain_psku_june_live.xlsx',
 '/data/aman_singh/acuuracy_check/Norms 202606.csv',
 '/data/aman_singh/acuuracy_check/key_check.csv',
 '/data/aman_singh/acuuracy_check/Norms 202607.csv',
 '/data/aman_singh/acuuracy_check/all_combination_qcom_july_pred.csv',
 '/data/aman_singh/acuuracy_check/Stat Demand Forecast MT_as_on_11th_May_2026.xlsb',
 '/data/aman_singh/acuuracy_check/acc_framework_may_final.xlsx',
 '/data/aman_singh/acuuracy_check/acc_offtakes_till_may.csv',
 '/data/aman_singh/acuuracy_check/swigy_vol

In [6]:
def discover_channel(file_path):
    # file_path = file_path.split('/')

    # if 'ECOM' in file_path:
    #     return 'ECOM'
    # elif 'QCOM' in file_path:
    #     return 'QCOM'
    # elif 'MT' in file_path:
    #     return 'MT'
    # else:
    #     return 'Channel not found'

    return 'ECOM'


In [7]:
from maricovault.MaricoDB import MaricoSnowflake

def get_dbconnection(db_name):    

    KEY_VAULT_NAME = "prod-pwd"

    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'
    

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection


def read_qtr_ind_rate_table():
    """
    Fetch the club sku information from  DWH_SAP_INDEX_TURNOVER_MONTHWISE table.

    Return:
        qtr_ind_rate_data: pandas dataframe
        - dataframe contains all the results from the index rate table.
    """
    connection = get_dbconnection(db_name='PROD')
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=connection, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    connection.close()
    return qtr_ind_rate


dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.

Credentials retrieved successfully for prod db.


In [8]:
data_query = f"""
    select * from {input_table}
    where month_date >= '2023-01-31' and run_month = '{run_month}'
        
"""

offtake_df = pd.read_sql(data_query, dev_conn)
offtake_df.head()

,MONTH_DATE,KEY,PLATFORM_NAME,DEPOT,PARENT_MATERIAL_CODE,BRAND_CODE,VOL_IN_RUM,IMPUTED,RUN_MONTH
0,2024-05-31,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.145,0,2026-08-31
1,2024-06-30,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,1,2026-08-31
2,2024-07-31,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,1,2026-08-31
3,2024-08-31,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,1,2026-08-31
4,2024-09-30,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,1,2026-08-31


In [9]:
offtake_df.columns = offtake_df.columns.str.lower()

In [10]:
# offtake_df = pd.read_csv('/data/aman_singh/mt_forecast/qcom_depot_psku_data.csv')
# offtake_df = offtake_df[offtake_df['month_date']<='2026-03-31']
# offtake_df

In [11]:
offtake_df.duplicated(
    subset=['platform_name','depot','parent_material_code', 'month_date']).sum()

0

In [12]:
offtake_df['brand_code'] = np.where(
    ((offtake_df['parent_material_code'] == 715096) &
    (offtake_df['brand_code'] == 'CO_SO_PCP')),
    'CO_SO_FS',
    offtake_df['brand_code']
)

In [13]:
# offtake_df = offtake_df[offtake_df['platform_name'].isin(
#     ['Amazon', 'Big Basket', 'Flipkart Grocery', 'Flipkart National'])]

In [14]:
offtake_df['key'] = offtake_df[['platform_name','depot','parent_material_code']].astype(str).agg('_'.join, axis=1)
# offtake_df.rename(columns={'realigned_psku': 'parent_material_code'}, inplace=True)
# offtake_df.drop([ 'run_month'], axis=1, inplace=True)
offtake_df['parent_material_code'] = offtake_df['parent_material_code'].astype(int)

In [15]:
offtake_df.duplicated(subset=['key', 'month_date']).sum()

0

In [16]:
(offtake_df['key'] == offtake_df[['platform_name','parent_material_code']].astype(str).agg('_'.join, axis=1)).all()

False

In [17]:
# realigned_df.to_csv('OT_data_debug.csv', index=False)

### Collate MIL

In [18]:
base_dir

'/data/aman_singh/acuuracy_check'

In [19]:
def collate_file(file_hint, extension='.csv'):
    collated_file = pd.DataFrame()

    run_path = f'{base_dir}'
    all_files = list_all_files_in_directory(run_path)

    for file_path in all_files:
        if file_hint in file_path:
            if extension == '.csv':
                print(file_path)
                read_file = pd.read_csv(file_path)
                # read_file['channel'] = discover_channel(file_path)
                read_file['run'] = 'run'
                read_file['step'] = file_path.split('/')[3]
                read_file['file_path'] = file_path

                collated_file = pd.concat(
                    [collated_file, read_file]
                )
                del read_file

    return collated_file

In [20]:
trend_file_df = collate_file('trend_file_train_till')
prophet_file_df = collate_file('prophet_data_train_till')

/data/aman_singh/acuuracy_check/trend_file_train_till_31_Jul_2026 (2).csv
/data/aman_singh/acuuracy_check/trend_file_train_till_31_Jul_2026 (3).csv
/data/aman_singh/acuuracy_check/trend_file_train_till_31_Jul_2026 (4).csv
/data/aman_singh/acuuracy_check/prophet_data_train_till_31_Jul_2026 (3).csv
/data/aman_singh/acuuracy_check/prophet_data_train_till_31_Jul_2026 (4).csv
/data/aman_singh/acuuracy_check/prophet_data_train_till_31_Jul_2026 (2).csv


In [21]:
# forecast_train_till_file_df = collate_file('forecast_train_till_')

In [22]:
# forecast_train_till_file_df

In [23]:
trend_file_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path
0,swiggy_D112_715101,2023-01-31,1.600,0.800,1.276554,3.154286,0.000195,0.000098,0.000156,0.000385,...,0.000586,NaN,NaN,4.8,0.000586,2026-07-31,6.557439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
1,swiggy_D112_715101,2023-02-28,1.600,0.800,0.360877,0.960000,0.000195,0.000098,0.000044,0.000117,...,0.000000,NaN,NaN,0.0,0.000000,2026-07-31,6.557439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
2,swiggy_D112_715101,2023-03-31,1.600,0.800,0.151301,0.274286,0.000195,0.000098,0.000018,0.000033,...,0.000000,NaN,NaN,0.0,0.000000,2026-07-31,6.557439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
3,swiggy_D112_715101,2023-04-30,1.600,0.800,0.341322,0.411429,0.000195,0.000098,0.000042,0.000050,...,0.000000,NaN,NaN,0.0,0.000000,2026-07-31,6.557439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
4,swiggy_D112_715101,2023-05-31,0.000,0.800,0.157659,0.205714,0.000000,0.000098,0.000019,0.000025,...,0.000000,NaN,NaN,0.0,0.000000,2026-07-31,6.557439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101877,blinkit_D677_810522,2026-08-31,0.002,0.006,0.004171,0.002749,0.000052,0.000156,0.000108,0.000071,...,0.000000,0.002749,0.000071,0.0,0.000000,2026-07-31,1.020325,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
101878,blinkit_D677_810522,2026-09-30,0.002,0.006,0.003625,0.003340,0.000052,0.000156,0.000094,0.000087,...,0.000000,0.003340,0.000087,0.0,0.000000,2026-07-31,1.020325,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
101879,blinkit_D677_810522,2026-10-31,0.002,0.006,0.004177,0.004043,0.000052,0.000156,0.000109,0.000105,...,0.000000,0.004043,0.000105,0.0,0.000000,2026-07-31,1.020325,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...
101880,blinkit_D677_810522,2026-11-30,0.002,0.006,0.008558,0.006291,0.000052,0.000156,0.000223,0.000164,...,0.000000,0.006291,0.000164,0.0,0.000000,2026-07-31,1.020325,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...


In [24]:
# prophet_file_df.to_csv('Prophet_file_OT_FK_AZ_BB_.csv', index=False)

In [25]:
trend_file_df.columns

Index(['key', 'month_date', 'pred_p3m', 'pred_p6m', 'pred_prophet', 'pred_rf',
       'pred_value_p3m', 'pred_value_p6m', 'pred_value_prophet',
       'pred_value_rf', 'parent_material_code', 'depot', 'platform_name',
       'vol_in_rum', 'brand_code', 'qtr_ind_rate', 'vol_in_rum_value',
       'pred_best_model', 'pred_value_best_model', 'vol_in_rum_treated',
       'vol_in_rum_value_treated', 'train_till', 'cov', 'run', 'step',
       'file_path'],
      dtype='object')

In [26]:
trend_file_df.dtypes

key                          object
month_date                   object
pred_p3m                    float64
pred_p6m                    float64
pred_prophet                float64
pred_rf                     float64
pred_value_p3m              float64
pred_value_p6m              float64
pred_value_prophet          float64
pred_value_rf               float64
parent_material_code          int64
depot                        object
platform_name                object
vol_in_rum                  float64
brand_code                   object
qtr_ind_rate                float64
vol_in_rum_value            float64
pred_best_model             float64
pred_value_best_model       float64
vol_in_rum_treated          float64
vol_in_rum_value_treated    float64
train_till                   object
cov                         float64
run                          object
step                         object
file_path                    object
dtype: object

In [27]:
trend_file_df['month_date'] = pd.to_datetime(trend_file_df['month_date'])
prophet_file_df['month_date'] = pd.to_datetime(prophet_file_df['month_date'])

trend_file_df['train_till'] = pd.to_datetime(trend_file_df['train_till'])
prophet_file_df['train_till'] = pd.to_datetime(prophet_file_df['train_till'])

trend_file_df['run_month'] = pd.to_datetime(trend_file_df['train_till'] + MonthEnd(1))
prophet_file_df['run_month'] = pd.to_datetime(prophet_file_df['train_till'] + MonthEnd(1))

In [28]:
trend_file_df.duplicated(subset=['key', 'month_date', 'run_month']).sum(), \
prophet_file_df.duplicated(subset=['key', 'month_date', 'run_month']).sum()

(0, 0)

In [29]:
mappings = {}

for run_month in trend_file_df['run_month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 9):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


mappings   

{Timestamp('2026-08-31 00:00:00'): {Timestamp('2026-08-31 00:00:00'): 'M',
  Timestamp('2026-09-30 00:00:00'): 'M+1',
  Timestamp('2026-10-31 00:00:00'): 'M+2',
  Timestamp('2026-11-30 00:00:00'): 'M+3',
  Timestamp('2026-12-31 00:00:00'): 'M+4',
  Timestamp('2027-01-31 00:00:00'): 'M+5',
  Timestamp('2027-02-28 00:00:00'): 'M+6',
  Timestamp('2027-03-31 00:00:00'): 'M+7',
  Timestamp('2027-04-30 00:00:00'): 'M+8'}}

In [30]:
trend_file_df['M month'] = trend_file_df.apply(
    lambda x: mappings[x['run_month']].get(
        x['month_date']
    ), axis=1
)

In [31]:
trend_file_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month
0,swiggy_D112_715101,2023-01-31,1.600,0.800,1.276554,3.154286,0.000195,0.000098,0.000156,0.000385,...,NaN,4.8,0.000586,2026-07-31,6.557439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None
1,swiggy_D112_715101,2023-02-28,1.600,0.800,0.360877,0.960000,0.000195,0.000098,0.000044,0.000117,...,NaN,0.0,0.000000,2026-07-31,6.557439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None
2,swiggy_D112_715101,2023-03-31,1.600,0.800,0.151301,0.274286,0.000195,0.000098,0.000018,0.000033,...,NaN,0.0,0.000000,2026-07-31,6.557439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None
3,swiggy_D112_715101,2023-04-30,1.600,0.800,0.341322,0.411429,0.000195,0.000098,0.000042,0.000050,...,NaN,0.0,0.000000,2026-07-31,6.557439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None
4,swiggy_D112_715101,2023-05-31,0.000,0.800,0.157659,0.205714,0.000000,0.000098,0.000019,0.000025,...,NaN,0.0,0.000000,2026-07-31,6.557439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101877,blinkit_D677_810522,2026-08-31,0.002,0.006,0.004171,0.002749,0.000052,0.000156,0.000108,0.000071,...,0.000071,0.0,0.000000,2026-07-31,1.020325,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,M
101878,blinkit_D677_810522,2026-09-30,0.002,0.006,0.003625,0.003340,0.000052,0.000156,0.000094,0.000087,...,0.000087,0.0,0.000000,2026-07-31,1.020325,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,M+1
101879,blinkit_D677_810522,2026-10-31,0.002,0.006,0.004177,0.004043,0.000052,0.000156,0.000109,0.000105,...,0.000105,0.0,0.000000,2026-07-31,1.020325,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,M+2
101880,blinkit_D677_810522,2026-11-30,0.002,0.006,0.008558,0.006291,0.000052,0.000156,0.000223,0.000164,...,0.000164,0.0,0.000000,2026-07-31,1.020325,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,M+3


In [32]:
trend_file_df[trend_file_df['M month'].notna()]

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month
43,swiggy_D112_715101,2026-08-31,0.000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.0,2026-07-31,6.557439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,M
44,swiggy_D112_715101,2026-09-30,0.000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.0,2026-07-31,6.557439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,M+1
45,swiggy_D112_715101,2026-10-31,0.000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.0,2026-07-31,6.557439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,M+2
46,swiggy_D112_715101,2026-11-30,0.000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.0,2026-07-31,6.557439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,M+3
47,swiggy_D112_715101,2026-12-31,0.000,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.0,2026-07-31,6.557439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,M+4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101877,blinkit_D677_810522,2026-08-31,0.002,0.006,0.004171,0.002749,0.000052,0.000156,0.000108,0.000071,...,0.000071,0.0,0.0,2026-07-31,1.020325,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,M
101878,blinkit_D677_810522,2026-09-30,0.002,0.006,0.003625,0.003340,0.000052,0.000156,0.000094,0.000087,...,0.000087,0.0,0.0,2026-07-31,1.020325,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,M+1
101879,blinkit_D677_810522,2026-10-31,0.002,0.006,0.004177,0.004043,0.000052,0.000156,0.000109,0.000105,...,0.000105,0.0,0.0,2026-07-31,1.020325,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,M+2
101880,blinkit_D677_810522,2026-11-30,0.002,0.006,0.008558,0.006291,0.000052,0.000156,0.000223,0.000164,...,0.000164,0.0,0.0,2026-07-31,1.020325,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,M+3


In [33]:
trend_file_df[['run_month', 'train_till']].drop_duplicates().sort_values(by=['run_month'])

,run_month,train_till
0,2026-08-31,2026-07-31


In [34]:
prophet_file_df[['run_month', 'train_till']].drop_duplicates().sort_values(by=['run_month'])

,run_month,train_till
0,2026-08-31,2026-07-31


In [35]:
trend_file_df['M month'].unique()

array([None, 'M', 'M+1', 'M+2', 'M+3', 'M+4'], dtype=object)

In [36]:
brand_md_df = pd.read_excel(r"/data/aman_singh/mt_forecast/Brand_metadata.xlsx")

In [37]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    brand_md_df,
    on=['brand_code'],
    how='left'
)
assert len_before_merge == len(trend_file_df)

In [38]:
trend_file_df['portfolio'].isna().sum()

0

In [39]:
prophet_file_df[
    ['month_date', 'key', 'run_month']
].duplicated().sum()

0

In [40]:
prophet_file_df

,ds,trend,yhat_lower,yhat_upper,trend_lower,trend_upper,yhat_60_%ile,yhat_70_%ile,yhat_75_%ile,trend_60_%ile,...,vol_in_rum_value,yhat_value,Model_Run,Model_Type,type,train_till,run,step,file_path,run_month
0,2023-01-31,0.005443,0.000563,0.016506,0.005443,0.005443,0.010428,0.012163,0.013126,0.005443,...,0.000360,0.000111,Yes,prophet,training,2026-07-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,2026-08-31
1,2023-02-28,0.005269,-0.002408,0.012603,0.005269,0.005269,0.006407,0.008242,0.009131,0.005269,...,0.000000,0.000063,Yes,prophet,training,2026-07-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,2026-08-31
2,2023-03-31,0.005075,0.003363,0.018849,0.005075,0.005075,0.012541,0.014319,0.015379,0.005075,...,0.000508,0.000145,Yes,prophet,training,2026-07-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,2026-08-31
3,2023-04-30,0.004888,-0.002777,0.012489,0.004888,0.004888,0.006316,0.007948,0.008707,0.004888,...,0.000000,0.000061,Yes,prophet,training,2026-07-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,2026-08-31
4,2023-05-31,0.004694,-0.005800,0.009369,0.004694,0.004694,0.003365,0.005199,0.005944,0.004694,...,0.000000,0.000030,Yes,prophet,training,2026-07-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,2026-08-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116048,2026-08-31,0.279061,0.192842,0.291677,0.279061,0.279061,0.252365,0.262872,0.268262,0.279061,...,0.000000,0.000311,Yes,prophet,testing,2026-07-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,2026-08-31
116049,2026-09-30,0.291903,0.294522,0.396490,0.291903,0.291903,0.355286,0.365894,0.372558,0.291903,...,0.000000,0.000445,Yes,prophet,testing,2026-07-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,2026-08-31
116050,2026-10-31,0.305174,0.279999,0.383806,0.305174,0.305174,0.342166,0.351767,0.360036,0.305174,...,0.000000,0.000427,Yes,prophet,testing,2026-07-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,2026-08-31
116051,2026-11-30,0.318017,0.177855,0.284278,0.318017,0.318017,0.240864,0.250921,0.256724,0.318017,...,0.000000,0.000294,Yes,prophet,testing,2026-07-31,run,acuuracy_check,/data/aman_singh/acuuracy_check/prophet_data_t...,2026-08-31


In [41]:
# Merge 70th percentile Prophet predictions
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    prophet_file_df[['month_date', 'key', 'run_month', 'yhat_70_%ile', 'yhat_60_%ile']].rename(
        columns={
            'yhat_70_%ile': 'pred_prophet_70%ile',
            'yhat_60_%ile': 'pred_prophet_60%ile'
        }
    ),
    on=['month_date', 'key', 'run_month'],
    how='left'
)
assert len(trend_file_df) == len_before_merge

In [42]:
assert trend_file_df.duplicated(
    subset=['run_month', 'month_date', 'key']
).sum() == 0

In [43]:
trend_file_df.drop('vol_in_rum', axis=1, inplace=True)

In [44]:
offtake_df.head()

,month_date,key,platform_name,depot,parent_material_code,brand_code,vol_in_rum,imputed,run_month
0,2024-05-31,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.145,0,2026-08-31
1,2024-06-30,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,1,2026-08-31
2,2024-07-31,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,1,2026-08-31
3,2024-08-31,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,1,2026-08-31
4,2024-09-30,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,1,2026-08-31


In [45]:
assert offtake_df.duplicated(subset=['key', 'month_date']).sum() == 0

In [46]:
offtake_df['month_date'] = pd.to_datetime(offtake_df['month_date'])
# offtake_df['run_month'] = pd.to_datetime(offtake_df['run_month'])

In [47]:
len_before_merge = len(trend_file_df)
trend_file_df = trend_file_df.merge(
    offtake_df[['key','month_date', 'vol_in_rum']],
    on=['month_date', 'key'],
    how='left'
)
assert len(trend_file_df) == len_before_merge

In [48]:
trend_file_df.select_dtypes('number').isna().sum()

pred_p3m                         0
pred_p6m                         0
pred_prophet                     0
pred_rf                          0
pred_value_p3m                   0
pred_value_p6m                   0
pred_value_prophet               0
pred_value_rf                    0
parent_material_code             0
qtr_ind_rate                     0
vol_in_rum_value                 0
pred_best_model             237687
pred_value_best_model       237687
vol_in_rum_treated               0
vol_in_rum_value_treated         0
cov                              0
pred_prophet_70%ile              0
pred_prophet_60%ile              0
vol_in_rum                       0
dtype: int64

In [49]:
trend_file_df.select_dtypes('number').min().round()

pred_p3m                         0.0
pred_p6m                         0.0
pred_prophet                     0.0
pred_rf                          0.0
pred_value_p3m                   0.0
pred_value_p6m                   0.0
pred_value_prophet               0.0
pred_value_rf                    0.0
parent_material_code        715101.0
qtr_ind_rate                   100.0
vol_in_rum_value                 0.0
pred_best_model                  0.0
pred_value_best_model            0.0
vol_in_rum_treated               0.0
vol_in_rum_value_treated         0.0
cov                              0.0
pred_prophet_70%ile          -1286.0
pred_prophet_60%ile          -1605.0
vol_in_rum                       0.0
dtype: float64

In [50]:
trend_file_df['vol_in_rum'].fillna(0, inplace=True)

In [51]:
for col in [ 'pred_best_model', 'pred_value_best_model', 'pred_prophet_70%ile','pred_prophet_60%ile', 'vol_in_rum']:
    trend_file_df[col] = trend_file_df[col].clip(lower=0)

In [52]:
trend_file_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum
0,swiggy_D112_715101,2023-01-31,1.600,0.800,1.276554,3.154286,0.000195,0.000098,0.000156,0.000385,...,6.557439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Skin Care,1.607524,1.424773,4.8
1,swiggy_D112_715101,2023-02-28,1.600,0.800,0.360877,0.960000,0.000195,0.000098,0.000044,0.000117,...,6.557439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Skin Care,0.661395,0.523395,0.0
2,swiggy_D112_715101,2023-03-31,1.600,0.800,0.151301,0.274286,0.000195,0.000098,0.000018,0.000033,...,6.557439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Skin Care,0.445290,0.302572,0.0
3,swiggy_D112_715101,2023-04-30,1.600,0.800,0.341322,0.411429,0.000195,0.000098,0.000042,0.000050,...,6.557439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Skin Care,0.680217,0.478930,0.0
4,swiggy_D112_715101,2023-05-31,0.000,0.800,0.157659,0.205714,0.000000,0.000098,0.000019,0.000025,...,6.557439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Skin Care,0.459613,0.297013,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
280172,blinkit_D677_810522,2026-08-31,0.002,0.006,0.004171,0.002749,0.000052,0.000156,0.000108,0.000071,...,1.020325,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,M,Saffola Oils,0.004633,0.004384,0.0
280173,blinkit_D677_810522,2026-09-30,0.002,0.006,0.003625,0.003340,0.000052,0.000156,0.000094,0.000087,...,1.020325,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,M+1,Saffola Oils,0.004094,0.003854,0.0
280174,blinkit_D677_810522,2026-10-31,0.002,0.006,0.004177,0.004043,0.000052,0.000156,0.000109,0.000105,...,1.020325,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,M+2,Saffola Oils,0.004559,0.004350,0.0
280175,blinkit_D677_810522,2026-11-30,0.002,0.006,0.008558,0.006291,0.000052,0.000156,0.000223,0.000164,...,1.020325,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,M+3,Saffola Oils,0.008972,0.008755,0.0


In [53]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)

In [54]:
trend_file_df['P3M'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(1)\
                                .rolling(window=3, min_periods=3).mean()

trend_file_df['P6M'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(1)\
                                .rolling(window=6, min_periods=6).mean()

trend_file_df['LY P3M'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(13)\
                                .rolling(window=3, min_periods=3).mean()

trend_file_df['LY P6M'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(13)\
                                .rolling(window=6, min_periods=6).mean()

In [55]:
trend_file_df['LY P3M_copy'] = trend_file_df['LY P3M'].copy()

In [56]:
trend_file_df['P3M Max'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).max()

In [57]:
trend_file_df['P3M Top 2 Mean'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sort(x)[-2:].mean(), raw=True) 

# lambda x: x.nlargest(2).mean(), raw=False

In [58]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)

In [59]:
trend_file_df['MoM P3M growth'] = (
    trend_file_df.groupby(['run_month', 'key'])['P3M']
      .pct_change() * 100
)

In [60]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
trend_file_df['MoM P3M growth_lag_1'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(1)
trend_file_df['MoM P3M growth_lag_2'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(2)



In [61]:
trend_file_df['>=20%_3M_inc_month_count'] = trend_file_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['MoM P3M growth'].shift(0)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sum(x >= 20), raw=True) 

In [62]:
trend_file_df['Avg(P3M Mean, Max)'] = trend_file_df[['P3M', 'P3M Max']].mean(axis=1)

In [63]:
for col in ['P3M', 'P6M', 'LY P3M', 'P3M Max', 'P3M Top 2 Mean', 
            'MoM P3M growth', '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)',
            'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2' ]:
    # if not 'LY' in col:  'LY P6M',
    trend_file_df.loc[trend_file_df['month_date'] > trend_file_df['run_month'], [col]] = np.nan
    trend_file_df[col] = trend_file_df.groupby(['run_month','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )


# for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
#     if not 'LY' in col:
#         trend_file_df.loc[trend_file_df['month_date'] > trend_file_df['run_month'], [col]] = np.nan
#         trend_file_df[col] = trend_file_df.groupby(['key'], as_index = True, group_keys = False)[col].apply(
#             lambda x: x.ffill()
#         )

In [64]:
# trend_file_df.to_csv('collate_check.csv', index=False)

In [65]:
for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
    trend_file_df[f'{col}_value'] = trend_file_df[col] * trend_file_df['qtr_ind_rate'] / (10 ** 7)

In [66]:
trend_file_df['vol_in_rum_value'] = trend_file_df['qtr_ind_rate'] * trend_file_df['vol_in_rum'] / (10 ** 7)
trend_file_df['pred_prophet_70%ile_value'] = trend_file_df['qtr_ind_rate'] * trend_file_df['pred_prophet_70%ile'] / (10 ** 7)
trend_file_df['pred_prophet_60%ile_value'] = trend_file_df['qtr_ind_rate'] * trend_file_df['pred_prophet_60%ile'] / (10 ** 7)

In [67]:
value_cols = [col for col in trend_file_df.columns if 'value' in col]
value_cols

['pred_value_p3m',
 'pred_value_p6m',
 'pred_value_prophet',
 'pred_value_rf',
 'vol_in_rum_value',
 'pred_value_best_model',
 'vol_in_rum_value_treated',
 'P3M_value',
 'P6M_value',
 'LY P3M_value',
 'LY P6M_value',
 'pred_prophet_70%ile_value',
 'pred_prophet_60%ile_value']

In [68]:
for col in value_cols:
    try:
        assert trend_file_df[col].min() >= 0
    except:
        print(col)
    

    # trend_file_df[col] = trend_file_df[col] / (10 ** 7)

In [69]:
trend_file_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value
179678,blinkit_D112_718589,2024-05-31,0.5,0.25,0.627385,1.151,0.000025,0.000012,3.117025e-05,0.000057,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.468066e-05,3.929947e-05
179679,blinkit_D112_718589,2024-06-30,0.5,0.25,0.000000,1.061,0.000025,0.000012,0.000000e+00,0.000053,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.396204e-06,0.000000e+00
179680,blinkit_D112_718589,2024-07-31,0.5,0.25,0.383315,0.145,0.000025,0.000012,1.904420e-05,0.000007,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.035701e-05,2.475381e-05
179681,blinkit_D112_718589,2024-08-31,0.5,0.25,0.482854,0.250,0.000025,0.000012,2.398958e-05,0.000012,...,NaN,NaN,NaN,0.5,0.000025,NaN,NaN,NaN,3.547156e-05,2.962657e-05
179682,blinkit_D112_718589,2024-09-30,0.0,0.25,0.085282,0.351,0.000000,0.000012,4.237064e-06,0.000017,...,NaN,NaN,NaN,0.0,0.000000,NaN,NaN,NaN,1.547185e-05,8.849673e-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76256,swiggy_D530_719192,2026-08-31,0.0,0.00,0.000000,0.000,0.000000,0.000000,0.000000e+00,0.000000,...,-100.0,-100.0,NaN,0.0,0.000000,0.0,0.0,0.0,4.401673e-08,0.000000e+00
76257,swiggy_D530_719192,2026-09-30,0.0,0.00,0.003720,0.000,0.000000,0.000000,3.720157e-08,0.000000,...,-100.0,-100.0,NaN,0.0,0.000000,0.0,0.0,0.0,1.529905e-07,8.677857e-08
76258,swiggy_D530_719192,2026-10-31,0.0,0.00,0.000000,0.000,0.000000,0.000000,0.000000e+00,0.000000,...,-100.0,-100.0,NaN,0.0,0.000000,0.0,0.0,0.0,7.376081e-08,2.028461e-08
76259,swiggy_D530_719192,2026-11-30,0.0,0.00,0.003828,0.000,0.000000,0.000000,3.827833e-08,0.000000,...,-100.0,-100.0,NaN,0.0,0.000000,0.0,0.0,0.0,1.423996e-07,8.358216e-08


In [70]:
assert trend_file_df.duplicated(
    subset=['run_month', 'brand_code', 'key', 'month_date']
).sum() == 0

In [71]:
trend_file_df = trend_file_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
trend_file_df['LY'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum'].shift(12)

trend_file_df['LLY'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum'].shift(24)


trend_file_df['LY value'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(12)

trend_file_df['LLY value'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(24)


trend_file_df['OT_Value_in_Cr_lag_1'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(1)

trend_file_df['OT_Value_in_Cr_lag_2'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(2)

trend_file_df['OT_Value_in_Cr_lag_3'] = trend_file_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(3)

In [72]:
for col in ['OT_Value_in_Cr_lag_1', 'OT_Value_in_Cr_lag_2', 'OT_Value_in_Cr_lag_3']:
    # if not 'LY' in col:  'LY P6M',
    trend_file_df.loc[trend_file_df['month_date'] > trend_file_df['run_month'], [col]] = np.nan
    trend_file_df[col] = trend_file_df.groupby(['run_month','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

In [73]:
trend_file_df.columns

Index(['key', 'month_date', 'pred_p3m', 'pred_p6m', 'pred_prophet', 'pred_rf',
       'pred_value_p3m', 'pred_value_p6m', 'pred_value_prophet',
       'pred_value_rf', 'parent_material_code', 'depot', 'platform_name',
       'brand_code', 'qtr_ind_rate', 'vol_in_rum_value', 'pred_best_model',
       'pred_value_best_model', 'vol_in_rum_treated',
       'vol_in_rum_value_treated', 'train_till', 'cov', 'run', 'step',
       'file_path', 'run_month', 'M month', 'portfolio', 'pred_prophet_70%ile',
       'pred_prophet_60%ile', 'vol_in_rum', 'P3M', 'P6M', 'LY P3M', 'LY P6M',
       'LY P3M_copy', 'P3M Max', 'P3M Top 2 Mean', 'MoM P3M growth',
       'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2',
       '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)', 'P3M_value',
       'P6M_value', 'LY P3M_value', 'LY P6M_value',
       'pred_prophet_70%ile_value', 'pred_prophet_60%ile_value', 'LY', 'LLY',
       'LY value', 'LLY value', 'OT_Value_in_Cr_lag_1', 'OT_Value_in_Cr_lag_2',
       'OT_Value_in

In [74]:
# trend_file_df[['ASM', 'Depot', 'PSKU']] = trend_file_df['key'].str.split('_', expand=True)

In [75]:
trend_file_df.reset_index(drop=True, inplace=True)

In [76]:
trend_file_df.shape

(280177, 56)

In [77]:
trend_file_df['key'].nunique()

8498

In [78]:
# batch_info = pd.read_excel(
#     '/data/aniket/az_demand_forecasting-mil-sc/channel_wise_batch.xlsx'
# )

In [79]:
# brand_class = pd.read_excel('/data/aman_singh/acuuracy_check/brand_class_new.xlsx')
# brand_class['Channel'] = brand_class['Channel'].replace({
#     'E-Commerce': 'ECOM', 'Q-Commerce': 'QCOM'})
# brand_class.columns = brand_class.columns.str.lower()
# brand_class = brand_class[brand_class['channel'] == 'QCOM']
# brand_class = brand_class[['brand','final class']]

# brand_class.rename(columns = {'brand':'brand_code','final class':'class'}, inplace = True)


In [80]:
# trend_file_df.drop(columns = ['final class'], inplace = True)

In [81]:
# brand_class

In [82]:
# len_before_merge = len(trend_file_df)
# trend_file_df = trend_file_df.merge(
#     brand_class, 
#     on=['brand_code'],
#     how='left'
# )
# assert len_before_merge == len(trend_file_df)
# del len_before_merge

In [83]:
# trend_file_df['class'].isna().sum()

In [84]:
# trend_file_df['class'].unique()

In [85]:
trend_file_df['run_month'].unique()

<DatetimeArray>
['2026-08-31 00:00:00']
Length: 1, dtype: datetime64[ns]

In [86]:
# trend_file_df[
#     # (trend_file_df['channel'].isin(['MT', 'QCOM'])) & 
#     # (trend_file_df['month_date'] > '2024-06-30') &
#     (trend_file_df['M month'].notna())
#     # (trend_file_df['class'].isin(['B', 'C']))
# ].to_csv('Heuristic_QCOM_Chain_PSKU_Offtakes_live2.csv', index=False)

missing combinations

In [87]:
model_file = trend_file_df.copy()

In [88]:
model_file['key'].nunique()

8498

In [89]:
run_month

Timestamp('2026-08-31 00:00:00')

In [90]:
data_query = f"""select * from {input_table} where month_date >= '2023-01-31' 
                and run_month = '2026-08-31' """
qcom_df = pd.read_sql(data_query, dev_conn)
qcom_df.head()

,MONTH_DATE,KEY,PLATFORM_NAME,DEPOT,PARENT_MATERIAL_CODE,BRAND_CODE,VOL_IN_RUM,IMPUTED,RUN_MONTH
0,2024-05-31,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.145,0,2026-08-31
1,2024-06-30,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,1,2026-08-31
2,2024-07-31,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,1,2026-08-31
3,2024-08-31,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,1,2026-08-31
4,2024-09-30,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,1,2026-08-31


In [91]:
qcom_df.columns = qcom_df.columns.str.lower()

In [92]:
qcom_df['key'] = qcom_df[['platform_name', 'depot','parent_material_code']].astype(str).agg('_'.join, axis=1)

In [93]:
qcom_df

,month_date,key,platform_name,depot,parent_material_code,brand_code,vol_in_rum,imputed,run_month
0,2024-05-31,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.145,0,2026-08-31
1,2024-06-30,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,1,2026-08-31
2,2024-07-31,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,1,2026-08-31
3,2024-08-31,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,1,2026-08-31
4,2024-09-30,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.000,1,2026-08-31
...,...,...,...,...,...,...,...,...,...
342948,2026-11-30,zepto_D677_810439,zepto,D677,810439,SAF-MUSLI,0.000,1,2026-08-31
342949,2026-12-31,zepto_D677_810439,zepto,D677,810439,SAF-MUSLI,0.000,1,2026-08-31
342950,2027-01-31,zepto_D677_810439,zepto,D677,810439,SAF-MUSLI,0.000,1,2026-08-31
342951,2027-02-28,zepto_D677_810439,zepto,D677,810439,SAF-MUSLI,0.000,1,2026-08-31


In [94]:
model_file['run_month'] = pd.to_datetime(model_file['run_month'])
model_file['month_date'] = pd.to_datetime(model_file['month_date'])

qcom_df['run_month'] = pd.to_datetime(qcom_df['run_month'])
qcom_df['month_date'] = pd.to_datetime(qcom_df['month_date'])

In [95]:
qcom_df.shape

(342953, 9)

In [96]:
tmp_df = model_file.groupby(['key', 'run_month'])['LY'].count().reset_index()
tmp_df#.isnull().sum()
#qcom_df[~qcom_df['key'].isin(model_file['key'].unique())]
qcom_df = qcom_df.merge(tmp_df, on = ['key','run_month'], how = 'left')
missing_df = qcom_df[qcom_df['LY'].isna()]
missing_df


,month_date,key,platform_name,depot,parent_material_code,brand_code,vol_in_rum,imputed,run_month,LY
1514,2025-10-31,blinkit_D112_718592,blinkit,D112,718592,SFOATS-FL,0.0055,0,2026-08-31,NaN
1515,2025-11-30,blinkit_D112_718592,blinkit,D112,718592,SFOATS-FL,0.0185,0,2026-08-31,NaN
1516,2025-12-31,blinkit_D112_718592,blinkit,D112,718592,SFOATS-FL,0.0250,0,2026-08-31,NaN
1517,2026-01-31,blinkit_D112_718592,blinkit,D112,718592,SFOATS-FL,0.0290,0,2026-08-31,NaN
1518,2026-02-28,blinkit_D112_718592,blinkit,D112,718592,SFOATS-FL,0.0275,0,2026-08-31,NaN
...,...,...,...,...,...,...,...,...,...,...
336703,2026-11-30,zepto_D674_811181,zepto,D674,811181,SAF_CDPRS,0.0000,1,2026-08-31,NaN
336704,2026-12-31,zepto_D674_811181,zepto,D674,811181,SAF_CDPRS,0.0000,1,2026-08-31,NaN
336705,2027-01-31,zepto_D674_811181,zepto,D674,811181,SAF_CDPRS,0.0000,1,2026-08-31,NaN
336706,2027-02-28,zepto_D674_811181,zepto,D674,811181,SAF_CDPRS,0.0000,1,2026-08-31,NaN


In [97]:
missing_df['key'].nunique()

2506

In [98]:
trend_file_df['key'].nunique()

8498

In [99]:
missing_df.isnull().sum()

month_date                  0
key                         0
platform_name               0
depot                       0
parent_material_code        0
brand_code                  0
vol_in_rum                  0
imputed                     0
run_month                   0
LY                      37282
dtype: int64

In [100]:
missing_df['key'].nunique()

2506

In [101]:

missing_df = missing_df[['key','run_month','month_date', 'platform_name', 'depot','parent_material_code', 'brand_code',
       'vol_in_rum']]
missing_df

,key,run_month,month_date,platform_name,depot,parent_material_code,brand_code,vol_in_rum
1514,blinkit_D112_718592,2026-08-31,2025-10-31,blinkit,D112,718592,SFOATS-FL,0.0055
1515,blinkit_D112_718592,2026-08-31,2025-11-30,blinkit,D112,718592,SFOATS-FL,0.0185
1516,blinkit_D112_718592,2026-08-31,2025-12-31,blinkit,D112,718592,SFOATS-FL,0.0250
1517,blinkit_D112_718592,2026-08-31,2026-01-31,blinkit,D112,718592,SFOATS-FL,0.0290
1518,blinkit_D112_718592,2026-08-31,2026-02-28,blinkit,D112,718592,SFOATS-FL,0.0275
...,...,...,...,...,...,...,...,...
336703,zepto_D674_811181,2026-08-31,2026-11-30,zepto,D674,811181,SAF_CDPRS,0.0000
336704,zepto_D674_811181,2026-08-31,2026-12-31,zepto,D674,811181,SAF_CDPRS,0.0000
336705,zepto_D674_811181,2026-08-31,2027-01-31,zepto,D674,811181,SAF_CDPRS,0.0000
336706,zepto_D674_811181,2026-08-31,2027-02-28,zepto,D674,811181,SAF_CDPRS,0.0000


In [102]:
qtr_df = read_qtr_ind_rate_table()
qtr_df.columns = qtr_df.columns.str.lower()
qtr_df.head()




Credentials retrieved successfully for prod db.


,month_date,brand_code,qtr_ind_rate
0,2027-03-31,PA_CN_HGO,488.152
1,2027-03-31,TRU_RAWDF,800.000
2,2027-03-31,TRU_PDRFR,850.570
3,2027-03-31,TRU_OATS,177.070
4,2027-03-31,TRU_QUINO,204.750


In [103]:
len_before_merge = len(missing_df)

missing_df = missing_df.rename(columns={'material_group_code': 'brand_code'}).merge(
    qtr_df.drop('month_date', axis=1),
    on=['brand_code'],
    how='left'
)

assert len_before_merge == len(missing_df)

In [104]:
missing_df

,key,run_month,month_date,platform_name,depot,parent_material_code,brand_code,vol_in_rum,qtr_ind_rate
0,blinkit_D112_718592,2026-08-31,2025-10-31,blinkit,D112,718592,SFOATS-FL,0.0055,292663.137458
1,blinkit_D112_718592,2026-08-31,2025-11-30,blinkit,D112,718592,SFOATS-FL,0.0185,292663.137458
2,blinkit_D112_718592,2026-08-31,2025-12-31,blinkit,D112,718592,SFOATS-FL,0.0250,292663.137458
3,blinkit_D112_718592,2026-08-31,2026-01-31,blinkit,D112,718592,SFOATS-FL,0.0290,292663.137458
4,blinkit_D112_718592,2026-08-31,2026-02-28,blinkit,D112,718592,SFOATS-FL,0.0275,292663.137458
...,...,...,...,...,...,...,...,...,...
37277,zepto_D674_811181,2026-08-31,2026-11-30,zepto,D674,811181,SAF_CDPRS,0.0000,260000.000000
37278,zepto_D674_811181,2026-08-31,2026-12-31,zepto,D674,811181,SAF_CDPRS,0.0000,260000.000000
37279,zepto_D674_811181,2026-08-31,2027-01-31,zepto,D674,811181,SAF_CDPRS,0.0000,260000.000000
37280,zepto_D674_811181,2026-08-31,2027-02-28,zepto,D674,811181,SAF_CDPRS,0.0000,260000.000000


In [105]:
missing_df['month_date'] = pd.to_datetime(missing_df['month_date'])
missing_df['run_month'] = pd.to_datetime(missing_df['run_month'])


mappings = {}

for run_month in missing_df['run_month'].unique():
    # run_month = pd.to_datetime(run_month)

    if not run_month in mappings:
        mappings[run_month] = {}

    mappings[run_month][run_month] = 'M'

    for idx in range(1, 9):
        mappings[run_month][run_month + MonthEnd(idx)] = f"M+{idx}"


mappings   
missing_df['M month'] = missing_df.apply(
    lambda x: mappings[x['run_month']].get(
        x['month_date']
    ), axis=1
)

brand_md_df = pd.read_excel(r"/data/aman_singh/mt_forecast/Brand_metadata.xlsx")
len_before_merge = len(missing_df)
missing_df = missing_df.merge(
    brand_md_df,
    on=['brand_code'],
    how='left'
)
assert len_before_merge == len(missing_df)

# Merge 70th percentile Prophet predictions

assert missing_df.duplicated(
    subset=['run_month', 'month_date', 'key']
).sum() == 0
missing_df.drop('vol_in_rum', axis=1, inplace=True)




In [106]:

offtake_df['month_date'] = pd.to_datetime(offtake_df['month_date'])
assert offtake_df.duplicated(subset=['key', 'month_date']).sum() == 0
len_before_merge = len(missing_df)
missing_df = missing_df.merge(
    offtake_df[['key', 'month_date', 'vol_in_rum']],
    on=['month_date', 'key'],
    how='left'
)
assert len(missing_df) == len_before_merge

missing_df['vol_in_rum'].fillna(0, inplace=True)
for col in [ 'vol_in_rum']:
    missing_df[col] = missing_df[col].clip(lower=0)
missing_df = missing_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
missing_df['P3M'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(1)\
                                .rolling(window=3, min_periods=1).mean()

missing_df['P6M'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(1)\
                                .rolling(window=6, min_periods=6).mean()

missing_df['LY P3M'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(13)\
                                .rolling(window=3, min_periods=3).mean()

missing_df['LY P6M'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['vol_in_rum'].shift(13)\
                                .rolling(window=6, min_periods=6).mean()
missing_df['LY P3M_copy'] = missing_df['LY P3M'].copy()
missing_df['P3M Max'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).max()
missing_df['P3M Top 2 Mean'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['P3M'].shift(1)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sort(x)[-2:].mean(), raw=True) 

# lambda x: x.nlargest(2).mean(), raw=False
missing_df = missing_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
missing_df['MoM P3M growth'] = (
    missing_df.groupby(['run_month', 'key'])['P3M']
      .pct_change() * 100
)
missing_df = missing_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
missing_df['MoM P3M growth_lag_1'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(1)
missing_df['MoM P3M growth_lag_2'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['MoM P3M growth'].shift(2)


missing_df['>=20%_3M_inc_month_count'] = missing_df.groupby(['run_month', 'key'], as_index = False, group_keys = False)['MoM P3M growth'].shift(0)\
                                .rolling(window=3, min_periods=3).apply(lambda x: np.sum(x >= 20), raw=True) 
missing_df['Avg(P3M Mean, Max)'] = missing_df[['P3M', 'P3M Max']].mean(axis=1)
for col in ['P3M', 'P6M', 'LY P3M', 'P3M Max', 'P3M Top 2 Mean', 
            'MoM P3M growth', '>=20%_3M_inc_month_count', 'Avg(P3M Mean, Max)',
            'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2' ]:
    # if not 'LY' in col:  'LY P6M',
    missing_df.loc[missing_df['month_date'] > missing_df['run_month'], [col]] = np.nan
    missing_df[col] = missing_df.groupby(['run_month','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )


# for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
#     if not 'LY' in col:
#         missing_df.loc[missing_df['month_date'] > missing_df['run_month'], [col]] = np.nan
#         missing_df[col] = missing_df.groupby(['key'], as_index = True, group_keys = False)[col].apply(
#             lambda x: x.ffill()
#         )
# missing_df.to_csv('collate_check.csv', index=False)
for col in ['P3M', 'P6M', 'LY P3M', 'LY P6M']:
    missing_df[f'{col}_value'] = missing_df[col] * missing_df['qtr_ind_rate'] / (10 ** 7)
missing_df['vol_in_rum_value'] = missing_df['qtr_ind_rate'] * missing_df['vol_in_rum'] / (10 ** 7)
value_cols = [col for col in missing_df.columns if 'value' in col]
value_cols
for col in value_cols:
    try:
        assert missing_df[col].min() >= 0
    except:
        print(col)
    

    # missing_df[col] = missing_df[col] / (10 ** 7)
missing_df
assert missing_df.duplicated(
    subset=['run_month', 'brand_code', 'key', 'month_date']
).sum() == 0
missing_df = missing_df.sort_values(
    ['run_month', 'brand_code', 'key', 'month_date']
)
missing_df['LY'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum'].shift(12)

missing_df['LLY'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum'].shift(24)


missing_df['LY value'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(12)

missing_df['LLY value'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(24)


missing_df['OT_Value_in_Cr_lag_1'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(1)

missing_df['OT_Value_in_Cr_lag_2'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(2)

missing_df['OT_Value_in_Cr_lag_3'] = missing_df.groupby(
    ['run_month', 'brand_code', 'key']
)['vol_in_rum_value'].shift(3)
for col in ['OT_Value_in_Cr_lag_1', 'OT_Value_in_Cr_lag_2', 'OT_Value_in_Cr_lag_3']:
    # if not 'LY' in col:  'LY P6M',
    missing_df.loc[missing_df['month_date'] > missing_df['run_month'], [col]] = np.nan
    missing_df[col] = missing_df.groupby(['run_month','key'], as_index = True, group_keys = False)[col].apply(
        lambda x: x.ffill()
    )

missing_df.reset_index(drop=True, inplace=True)

# brand_class_df = pd.read_excel(r"/data/aman_singh/acuuracy_check/Brand_Classification.xlsb")
# brand_class_df.head()
# brand_class_df.columns = ['brand_code', 'class']

# len_before_merge = len(missing_df)
# missing_df = missing_df.merge(
#     brand_class, 
#     on=['brand_code'],
#     how='left'
# )
# assert len_before_merge == len(missing_df)
# del len_before_merge
# missing_df['class'].isna().sum()
# missing_df['class'].unique()

LY P3M_value


In [107]:
pd.set_option('display.max_columns', None)

In [108]:
# missing_df[
#     # (missing_df['channel'].isin(['MT', 'QCOM'])) & 
#     # (missing_df['month_date'] > '2024-06-30') &
#     (missing_df['M month'].notna())
#     # (missing_df['class'].isin(['B', 'C']))
# ].to_csv('missing_combinations_QCOM_Chain_city_PSKU_Offtakes.csv', index=False)

In [109]:
model_file

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,depot,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3
0,blinkit_D112_718589,2024-05-31,0.5,0.25,0.627385,1.151,0.000025,0.000012,3.117025e-05,0.000057,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000075,NaN,NaN,1.5,0.000075,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,0.899318,0.791007,1.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.468066e-05,3.929947e-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,blinkit_D112_718589,2024-06-30,0.5,0.25,0.000000,1.061,0.000025,0.000012,0.000000e+00,0.000053,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,NaN,NaN,0.0,0.000000,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,0.108613,0.000000,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.396204e-06,0.000000e+00,NaN,NaN,NaN,NaN,0.000075,NaN,NaN
2,blinkit_D112_718589,2024-07-31,0.5,0.25,0.383315,0.145,0.000025,0.000012,1.904420e-05,0.000007,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,NaN,NaN,0.0,0.000000,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,0.611016,0.498236,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.035701e-05,2.475381e-05,NaN,NaN,NaN,NaN,0.000000,0.000075,NaN
3,blinkit_D112_718589,2024-08-31,0.5,0.25,0.482854,0.250,0.000025,0.000012,2.398958e-05,0.000012,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,NaN,NaN,0.0,0.000000,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,0.713960,0.596314,0.0,0.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.5,0.000025,NaN,NaN,NaN,3.547156e-05,2.962657e-05,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000075
4,blinkit_D112_718589,2024-09-30,0.0,0.25,0.085282,0.351,0.000000,0.000012,4.237064e-06,0.000017,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,NaN,NaN,0.0,0.000000,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,0.311412,0.178123,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,-100.0,NaN,NaN,NaN,0.0,0.000000,NaN,NaN,NaN,1.547185e-05,8.849673e-06,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
280172,swiggy_D530_719192,2026-08-31,0.0,0.00,0.000000,0.000,0.000000,0.000000,0.000000e+00,0.000000,719192,D530,swiggy,VEG_CLEAN,100.000000,0.000000,0.0,0.0,0.0,0.000000,2026-07-31,6.324555,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,M,Health & Hygiene,0.004402,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-100.0,-100.0,-100.0,NaN,0.0,0.000000,0.0,0.0,0.0,4.401673e-08,0.000000e+00,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000
280173,swiggy_D530_719192,2026-09-30,0.0,0.00,0.003720,0.000,0.000000,0.000000,3.720157e-08,0.000000,719192,D530,swiggy,VEG_CLEAN,100.000000,0.000000,0.0,0.0,0.0,0.000000,2026-07-31,6.324555,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,M+1,Health & Hygiene,0.015299,0.008678,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-100.0,-100.0,-100.0,NaN,0.0,0.000000,0.0,0.0,0.0,1.5299

In [110]:
model_file['skipped'] = 0
missing_df['skipped'] = 1
final_df = pd.concat([model_file,missing_df])
final_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,depot,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped
0,blinkit_D112_718589,2024-05-31,0.5,0.25,0.627385,1.151,0.000025,0.000012,0.000031,0.000057,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000075,NaN,NaN,1.5,0.000075,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,0.899318,0.791007,1.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000045,0.000039,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,blinkit_D112_718589,2024-06-30,0.5,0.25,0.000000,1.061,0.000025,0.000012,0.000000,0.000053,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,NaN,NaN,0.0,0.000000,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,0.108613,0.000000,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000005,0.000000,NaN,NaN,NaN,NaN,0.000075,NaN,NaN,0
2,blinkit_D112_718589,2024-07-31,0.5,0.25,0.383315,0.145,0.000025,0.000012,0.000019,0.000007,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,NaN,NaN,0.0,0.000000,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,0.611016,0.498236,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000030,0.000025,NaN,NaN,NaN,NaN,0.000000,0.000075,NaN,0
3,blinkit_D112_718589,2024-08-31,0.5,0.25,0.482854,0.250,0.000025,0.000012,0.000024,0.000012,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,NaN,NaN,0.0,0.000000,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,0.713960,0.596314,0.0,0.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.50,0.000025,NaN,NaN,NaN,0.000035,0.000030,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000075,0
4,blinkit_D112_718589,2024-09-30,0.0,0.25,0.085282,0.351,0.000000,0.000012,0.000004,0.000017,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,NaN,NaN,0.0,0.000000,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,0.311412,0.178123,0.0,0.00,NaN,NaN,NaN,NaN,NaN,NaN,-100.0,NaN,NaN,NaN,0.00,0.000000,NaN,NaN,NaN,0.000015,0.000009,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37277,zepto_D674_810125,2026-11-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,810125,D674,zepto,SW_SGPRF,1712.605337,0.000000,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,2026-08-31,M+3,Male Grooming,NaN,NaN,0.0,0.76,NaN,NaN,NaN,NaN,NaN,NaN,280.0,inf,NaN,NaN,0.76,0.000130,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000192,0.000069,NaN,1
37278,zepto_D674_810125,2026-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,810125,D674,zepto,SW_SGPRF,1712.605337,0.000000,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,2026-08-31,M+4,Male Grooming,NaN,NaN,0.0,0.76,NaN,NaN,NaN,NaN,NaN,NaN,280.0,inf,NaN,NaN,0.76,0.000130,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000192,0.000069,NaN,1
37279,zepto_D674_810125,2027-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,810125,D674,zepto,SW_SGPRF,1712.605337,0.000000,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,2026-08-31,M+5,Male Grooming,NaN,NaN,0.0,0.76,NaN,NaN,NaN,NaN,NaN,NaN,280.0,inf,NaN,NaN,0.76,0.000130,NaN,NaN,NaN,

In [111]:
# final_df = trend_file_df.copy()

In [112]:
final_df[final_df.select_dtypes(include='number').columns] = final_df.select_dtypes(include='number').fillna(0)
final_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,depot,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped
0,blinkit_D112_718589,2024-05-31,0.5,0.25,0.627385,1.151,0.000025,0.000012,0.000031,0.000057,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000075,0.0,0.0,1.5,0.000075,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,0.899318,0.791007,1.5,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000045,0.000039,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0
1,blinkit_D112_718589,2024-06-30,0.5,0.25,0.000000,1.061,0.000025,0.000012,0.000000,0.000053,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,0.0,0.0,0.0,0.000000,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,0.108613,0.000000,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000005,0.000000,0.0,0.0,0.0,0.0,0.000075,0.000000,0.000000,0
2,blinkit_D112_718589,2024-07-31,0.5,0.25,0.383315,0.145,0.000025,0.000012,0.000019,0.000007,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,0.0,0.0,0.0,0.000000,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,0.611016,0.498236,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000030,0.000025,0.0,0.0,0.0,0.0,0.000000,0.000075,0.000000,0
3,blinkit_D112_718589,2024-08-31,0.5,0.25,0.482854,0.250,0.000025,0.000012,0.000024,0.000012,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,0.0,0.0,0.0,0.000000,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,0.713960,0.596314,0.0,0.50,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.50,0.000025,0.0,0.0,0.0,0.000035,0.000030,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000075,0
4,blinkit_D112_718589,2024-09-30,0.0,0.25,0.085282,0.351,0.000000,0.000012,0.000004,0.000017,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,0.0,0.0,0.0,0.000000,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,0.311412,0.178123,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,-100.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000015,0.000009,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37277,zepto_D674_810125,2026-11-30,0.0,0.00,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,810125,D674,zepto,SW_SGPRF,1712.605337,0.000000,0.0,0.0,0.0,0.000000,NaT,0.000000,NaN,NaN,NaN,2026-08-31,M+3,Male Grooming,0.000000,0.000000,0.0,0.76,0.0,0.0,0.0,0.0,0.0,0.0,280.0,inf,0.0,0.0,0.76,0.000130,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000192,0.000069,0.000000,1
37278,zepto_D674_810125,2026-12-31,0.0,0.00,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,810125,D674,zepto,SW_SGPRF,1712.605337,0.000000,0.0,0.0,0.0,0.000000,NaT,0.000000,NaN,NaN,NaN,2026-08-31,M+4,Male Grooming,0.000000,0.000000,0.0,0.76,0.0,0.0,0.0,0.0,0.0,0.0,280.0,inf,0.0,0.0,0.76,0.000130,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000192,0.000069,0.000000,1
37279,zepto_D674_810125,2027-01-31,0.0,0.00,0.000000,0.000,0.000000,0.000000,0.000000,0.

In [113]:
final_df['month_date'] = pd.to_datetime(final_df['month_date'])
final_df['run_month'] = pd.to_datetime(final_df['run_month'])
final_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,depot,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped
0,blinkit_D112_718589,2024-05-31,0.5,0.25,0.627385,1.151,0.000025,0.000012,0.000031,0.000057,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000075,0.0,0.0,1.5,0.000075,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,0.899318,0.791007,1.5,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000045,0.000039,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0
1,blinkit_D112_718589,2024-06-30,0.5,0.25,0.000000,1.061,0.000025,0.000012,0.000000,0.000053,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,0.0,0.0,0.0,0.000000,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,0.108613,0.000000,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000005,0.000000,0.0,0.0,0.0,0.0,0.000075,0.000000,0.000000,0
2,blinkit_D112_718589,2024-07-31,0.5,0.25,0.383315,0.145,0.000025,0.000012,0.000019,0.000007,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,0.0,0.0,0.0,0.000000,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,0.611016,0.498236,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000030,0.000025,0.0,0.0,0.0,0.0,0.000000,0.000075,0.000000,0
3,blinkit_D112_718589,2024-08-31,0.5,0.25,0.482854,0.250,0.000025,0.000012,0.000024,0.000012,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,0.0,0.0,0.0,0.000000,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,0.713960,0.596314,0.0,0.50,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.50,0.000025,0.0,0.0,0.0,0.000035,0.000030,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000075,0
4,blinkit_D112_718589,2024-09-30,0.0,0.25,0.085282,0.351,0.000000,0.000012,0.000004,0.000017,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,0.0,0.0,0.0,0.000000,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,0.311412,0.178123,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,-100.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000015,0.000009,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37277,zepto_D674_810125,2026-11-30,0.0,0.00,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,810125,D674,zepto,SW_SGPRF,1712.605337,0.000000,0.0,0.0,0.0,0.000000,NaT,0.000000,NaN,NaN,NaN,2026-08-31,M+3,Male Grooming,0.000000,0.000000,0.0,0.76,0.0,0.0,0.0,0.0,0.0,0.0,280.0,inf,0.0,0.0,0.76,0.000130,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000192,0.000069,0.000000,1
37278,zepto_D674_810125,2026-12-31,0.0,0.00,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,810125,D674,zepto,SW_SGPRF,1712.605337,0.000000,0.0,0.0,0.0,0.000000,NaT,0.000000,NaN,NaN,NaN,2026-08-31,M+4,Male Grooming,0.000000,0.000000,0.0,0.76,0.0,0.0,0.0,0.0,0.0,0.0,280.0,inf,0.0,0.0,0.76,0.000130,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000192,0.000069,0.000000,1
37279,zepto_D674_810125,2027-01-31,0.0,0.00,0.000000,0.000,0.000000,0.000000,0.000000,0.

In [114]:
# final_df[
#     # (final_df['channel'].isin(['MT', 'QCOM'])) & 
#     # (final_df['month_date'] > '2024-06-30') &
#     (final_df['M month'].notna())
#     # (missing_df['class'].isin(['B', 'C']))
# ].to_csv('all_combinations_QCOM_chain_PSKU_Offtakes_DEC2.csv', index=False)

In [115]:
# final_df.to_csv('heuristic_data_preprocessed_qcom_jan.csv')

In [116]:
# import pandas as pd
# final_df = pd.read_csv('/data/aman_singh/acuuracy_check/heuristic_data_preprocessed_qcom_cp_live_jan.csv')
# final_df

In [117]:
final_df[(final_df['M month'].notna())]#['key'].nunique()

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,depot,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped
27,blinkit_D112_718589,2026-08-31,3.2,2.2,3.894027,2.4185,0.000159,0.000109,0.000193,0.000120,718589,D112,blinkit,ADV-AHO-R,496.828458,0.0,3.894027,0.000193,0.0,0.0,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,M,Hair Oils,4.139678,4.009291,0.0,3.20,2.2,0.6,0.30,0.6,2.5,2.3,28.0,19.047619,75.0,2.0,2.85,0.000159,0.000109,0.00003,0.000015,0.000206,0.000199,2.1,0.0,0.000104,0.0,0.000209,0.000104,0.000164,0
28,blinkit_D112_718589,2026-09-30,3.2,2.2,3.828703,2.5110,0.000159,0.000109,0.000190,0.000125,718589,D112,blinkit,ADV-AHO-R,496.828458,0.0,3.828703,0.000190,0.0,0.0,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,M+1,Hair Oils,4.100991,3.963845,0.0,3.20,2.2,0.6,0.65,1.3,2.5,2.3,28.0,19.047619,75.0,2.0,2.85,0.000159,0.000109,0.00003,0.000032,0.000204,0.000197,2.1,0.0,0.000104,0.0,0.000209,0.000104,0.000164,0
29,blinkit_D112_718589,2026-10-31,3.2,2.2,4.095376,2.5785,0.000159,0.000109,0.000203,0.000128,718589,D112,blinkit,ADV-AHO-R,496.828458,0.0,4.095376,0.000203,0.0,0.0,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,M+2,Hair Oils,4.368709,4.249045,0.0,3.20,2.2,0.6,1.00,1.8,2.5,2.3,28.0,19.047619,75.0,2.0,2.85,0.000159,0.000109,0.00003,0.000050,0.000217,0.000211,1.8,0.0,0.000089,0.0,0.000209,0.000104,0.000164,0
30,blinkit_D112_718589,2026-11-30,3.2,2.2,4.004326,2.5035,0.000159,0.000109,0.000199,0.000124,718589,D112,blinkit,ADV-AHO-R,496.828458,0.0,4.004326,0.000199,0.0,0.0,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,M+3,Hair Oils,4.239678,4.103096,0.0,3.20,2.2,0.6,1.30,2.0,2.5,2.3,28.0,19.047619,75.0,2.0,2.85,0.000159,0.000109,0.00003,0.000065,0.000211,0.000204,2.1,0.0,0.000104,0.0,0.000209,0.000104,0.000164,0
31,blinkit_D112_718589,2026-12-31,3.2,2.2,4.693727,2.8485,0.000159,0.000109,0.000233,0.000142,718589,D112,blinkit,ADV-AHO-R,496.828458,0.0,4.693727,0.000233,0.0,0.0,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,M+4,Hair Oils,4.955652,4.806584,0.0,3.20,2.2,0.6,1.65,2.0,2.5,2.3,28.0,19.047619,75.0,2.0,2.85,0.000159,0.000109,0.00003,0.000082,0.000246,0.000239,3.0,0.0,0.000149,0.0,0.000209,0.000104,0.000164,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37277,zepto_D674_810125,2026-11-30,0.0,0.0,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,810125,D674,zepto,SW_SGPRF,1712.605337,0.0,0.000000,0.000000,0.0,0.0,NaT,0.000000,NaN,NaN,NaN,2026-08-31,M+3,Male Grooming,0.000000,0.000000,0.0,0.76,0.0,0.0,0.00,0.0,0.0,0.0,280.0,inf,0.0,0.0,0.76,0.000130,0.000000,0.00000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000192,0.000069,0.000000,1
37278,zepto_D674_810125,2026-12-31,0.0,0.0,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,810125,D674,zepto,SW_SGPRF,1712.605337,0.0,0.000000,0.000000,0.0,0.0,NaT,0.000000,NaN,NaN,NaN,2026-08-31,M+4,Male Grooming,0.000000,0.000000,0.0,0.76,0.0,0.0,0.00,0.0,0.0,0.0,280.0,inf,0.0,0.0,0.76,0.000130,0.00000

In [118]:
final_df[final_df['month_date'] == '2026-08-31']['vol_in_rum_value'].sum()

0.0

Heuristic new approac

In [119]:
# pip install pymannkendall

In [120]:
# final_df.to_csv('brek2.csv', index=False)

In [121]:
import pandas as pd
import numpy as np
import pymannkendall as mk

def detect_trend_for_group(df_grp):
    """
    Detect final trend flag and p3m_slope_flag separately.
    Must contain 'month_date', 'vol_in_rum', 'run_month'
    """

    # ---------- 1. Sort ----------
    df_grp = df_grp.sort_values("month_date")

    # ---------- 2. Identify run_month ----------
    run_month = df_grp["run_month"].max()

    # actual data = months < run_month
    df_actual = df_grp[df_grp["month_date"] < run_month]

    # if no actual data → no trend
    if df_actual.empty or len(df_actual) < 4:
        return pd.Series({"trend_flag": 0, "p3m_slope_flag": 0})

    # ---------- 3. MK Trend ----------
    series = df_actual["vol_in_rum_value"].astype(float)

    try:
        mk_result = mk.original_test(series)
        if mk_result.trend == "increasing":
            mk_trend = 1
        elif mk_result.trend == "decreasing":
            mk_trend = -1
        else:
            mk_trend = 0
    except:
        mk_trend = 0

    # ---------- 4. P3M Slope ----------
    # last 4 months → take last 3 with shift
    #shifted_series = series.shift(1).dropna()

    p3m_values = series.tail(3).values
    #print(p3m_values)

    if len(p3m_values) < 3:
        slope_flag = 0
    else:
        x = np.arange(3)
        slope = np.polyfit(x, p3m_values, 1)[0]
        slope_flag = 1 if slope > 0 else (-1 if slope < 0 else 0)
        

    return pd.Series({
        "trend_flag": mk_trend,
        "p3m_slope_flag": slope_flag
    })


# ---------------------------------------------------------
# APPLY ON ENTIRE DATASET
# ---------------------------------------------------------

# trend_df = final_df.groupby(
#     ["platform_name", "parent_material_code"]
# ).apply(detect_trend_for_group).reset_index()

# trend_df = final_df[final_df['key'] == 'Zepto_721898'].groupby(
#     ["platform_name", "parent_material_code", "run_month"]
# ).apply(detect_trend_for_group).reset_index()
trend_df = final_df.groupby(
    ["platform_name", "depot","parent_material_code", "run_month"]
).apply(detect_trend_for_group).reset_index()

In [122]:
trend_df["final_trend"] = np.where(
    (trend_df["trend_flag"] == 1) & (trend_df["p3m_slope_flag"] == 1), 1,
    np.where(
        (trend_df["trend_flag"] == -1) & (trend_df["p3m_slope_flag"] == -1), -1,
        0
    )
)
trend_df

,platform_name,depot,parent_material_code,run_month,trend_flag,p3m_slope_flag,final_trend
0,blinkit,D112,718288,2026-08-31,0,0,0
1,blinkit,D112,718310,2026-08-31,-1,0,0
2,blinkit,D112,718312,2026-08-31,1,1,1
3,blinkit,D112,718317,2026-08-31,-1,0,0
4,blinkit,D112,718321,2026-08-31,-1,0,0
...,...,...,...,...,...,...,...
10999,zepto,D677,810372,2026-08-31,0,0,0
11000,zepto,D677,810405,2026-08-31,0,0,0
11001,zepto,D677,810406,2026-08-31,-1,0,0
11002,zepto,D677,810407,2026-08-31,0,0,0


In [123]:
trend_df[trend_df['final_trend'] == 1]

,platform_name,depot,parent_material_code,run_month,trend_flag,p3m_slope_flag,final_trend
2,blinkit,D112,718312,2026-08-31,1,1,1
8,blinkit,D112,718328,2026-08-31,1,1,1
12,blinkit,D112,718371,2026-08-31,1,1,1
19,blinkit,D112,718464,2026-08-31,1,1,1
25,blinkit,D112,718526,2026-08-31,1,1,1
...,...,...,...,...,...,...,...
10673,zepto,D674,728767,2026-08-31,1,1,1
10715,zepto,D674,808719,2026-08-31,1,1,1
10740,zepto,D674,809950,2026-08-31,1,1,1
10742,zepto,D674,810009,2026-08-31,1,1,1


## detect seasonality

In [124]:
from statsmodels.tsa.stattools import acf
import numpy as np
import pandas as pd

def detect_yearly_seasonality(df_grp, threshold=0.3):
    """
    Detects yearly seasonality using ACF at lag=12 only.
    Uses vol_in_rum as the metric.
    """
    df_grp = df_grp.sort_values("month_date")
    run_month = df_grp["run_month"].max()

    # actual data = months < run_month
    df_actual = df_grp[df_grp["month_date"] < run_month]
    series = df_actual["vol_in_rum"].astype(float).values

    # Need at least 18 points to compare last year vs this year
    if len(series) < 18:
        return 0

    # Compute ACF up to lag-12
    acf_vals = acf(series, nlags=12, fft=False)

    lag12_acf = acf_vals[12]

    # absolute ACF because seasonal correlation can be negative as well
    if abs(lag12_acf) >= threshold:
        return 1
    else:
        return 0
    

seasonality_df = final_df.groupby(
    ["platform_name", "brand_code", 'run_month']
).apply(detect_yearly_seasonality).reset_index(name="seasonality_flag")

seasonality_df



,platform_name,brand_code,run_month,seasonality_flag
0,blinkit,ADV-AHO-R,2026-08-31,0
1,blinkit,BIO OILS,2026-08-31,0
2,blinkit,CO_SO_PCP,2026-08-31,0
3,blinkit,CO_SO_VCN,2026-08-31,0
4,blinkit,H&C,2026-08-31,0
...,...,...,...,...
236,zepto,SW HRGEL,2026-08-31,0
237,zepto,SW HSPRY,2026-08-31,0
238,zepto,SW STLDEO,2026-08-31,0
239,zepto,SW_HR_WAX,2026-08-31,0


In [125]:
seasonality_df[seasonality_df['seasonality_flag'] == 1]

,platform_name,brand_code,run_month,seasonality_flag
5,blinkit,H&C DFOIL,2026-08-31,1
9,blinkit,LIVON 2.0,2026-08-31,1
58,blinkit,SAF_MAYO,2026-08-31,1
62,blinkit,SFFT_VNGR,2026-08-31,1
69,blinkit,SF_MNMKHN,2026-08-31,1
71,blinkit,SF_SOYBRJ,2026-08-31,1
74,blinkit,SW NOGAS,2026-08-31,1
139,swiggy,SAF_FT_MR,2026-08-31,1
145,swiggy,SFFT_GMIS,2026-08-31,1
147,swiggy,SFFT_VNGR,2026-08-31,1


In [126]:
# seasonality_df.to_csv('seasonality_qcom.csv')

In [127]:
import numpy as np
import pandas as pd

def compute_thresholds(df_grp):
    """
    df_grp MUST contain:
    - month_date
    - vol_in_rum
    - run_month

    Returns: lower_threshold, upper_threshold, mean, std
    """

    df_grp = df_grp.sort_values("month_date")
    run_month = df_grp["run_month"].max()

    # --- Use ONLY actual data (strictly before run month)
    df_actual = df_grp[df_grp["month_date"] < run_month]

    series = df_actual["vol_in_rum_value"].astype(float).values

    # If no real data → return zeros
    if len(series) == 0:
        return pd.Series({
            "lower_threshold": 0,
            "upper_threshold": 0,
            "mean_value": 0,
            "std_value": 0
        })

    # --- Take last 12 months OR all available
    if len(series) > 12:
        series = series[-12:]

    mean_val = np.mean(series)
    std_val = np.std(series)

    # --- SPECIAL CASE: ≤3 data points
    if len(series) <= 3:
        lower = 0.5 * mean_val
        upper = 2 * mean_val

        return pd.Series({
            "lower_threshold": lower,
            "upper_threshold": upper,
            "mean_value": mean_val,
            "std_value": std_val
        })

    # --- Normal case (std can be zero also)
    lower = max(0,mean_val - 2*std_val)
    upper = mean_val + 3*std_val

    return pd.Series({
        "lower_threshold": lower,
        "upper_threshold": upper,
        "mean_value": mean_val,
        "std_value": std_val
    })

threshold_df = final_df.groupby(
    ["platform_name", "depot","parent_material_code", "run_month"]
).apply(compute_thresholds).reset_index()

threshold_df.head()


,platform_name,depot,parent_material_code,run_month,lower_threshold,upper_threshold,mean_value,std_value
0,blinkit,D112,718288,2026-08-31,0.000000,0.000000,0.000000,0.000000
1,blinkit,D112,718310,2026-08-31,0.000000,0.003047,0.000397,0.000883
2,blinkit,D112,718312,2026-08-31,0.004933,0.063175,0.028230,0.011648
3,blinkit,D112,718317,2026-08-31,0.000000,0.000000,0.000000,0.000000
4,blinkit,D112,718321,2026-08-31,0.000000,0.000000,0.000000,0.000000


In [128]:
final_df[final_df['key'] == 'blinkit_D112_718317']

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,depot,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped
18395,blinkit_D112_718317,2023-01-31,0.900000,4.766667,2.956726,3.985810,0.000035,1.849831e-04,1.147436e-04,1.546799e-04,718317,D112,blinkit,H&C,388.076436,0.000047,0.0,0.000000,1.2,0.000047,2026-07-31,2.480278,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,3.909797,3.423997,1.2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000e+00,0.000000,0.000000e+00,1.517300e-04,0.000133,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0
18396,blinkit_D112_718317,2023-02-28,0.900000,4.766667,2.820552,3.557476,0.000035,1.849831e-04,1.094590e-04,1.380573e-04,718317,D112,blinkit,H&C,388.076436,0.000043,0.0,0.000000,1.1,0.000043,2026-07-31,2.480278,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,3.914025,3.290380,1.1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000e+00,0.000000,0.000000e+00,1.518941e-04,0.000128,0.0,0.0,0.000000,0.000000,0.000047,0.000000,0.000000,0
18397,blinkit_D112_718317,2023-03-31,0.900000,4.766667,2.728622,2.914976,0.000035,1.849831e-04,1.058914e-04,1.131234e-04,718317,D112,blinkit,H&C,388.076436,0.000016,0.0,0.000000,0.4,0.000016,2026-07-31,2.480278,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,3.975855,3.392724,0.4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000e+00,0.000000,0.000000e+00,1.542936e-04,0.000132,0.0,0.0,0.000000,0.000000,0.000043,0.000047,0.000000,0
18398,blinkit_D112_718317,2023-04-30,0.900000,4.766667,4.786135,7.175310,0.000035,1.849831e-04,1.857386e-04,2.784569e-04,718317,D112,blinkit,H&C,388.076436,0.000407,0.0,0.000000,10.5,0.000407,2026-07-31,2.480278,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,5.857375,5.244671,10.5,0.900000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.900000,0.000035,0.000000e+00,0.000000,0.000000e+00,2.273109e-04,0.000204,0.0,0.0,0.000000,0.000000,0.000016,0.000043,0.000047,0
18399,blinkit_D112_718317,2023-05-31,4.000000,4.766667,4.923150,6.214167,0.000155,1.849831e-04,1.910558e-04,2.411572e-04,718317,D112,blinkit,H&C,388.076436,0.000427,0.0,0.000000,11.0,0.000427,2026-07-31,2.480278,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,6.012420,5.439222,11.0,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,344.444444,0.000000,0.000000,0.0,4.000000,0.000155,0.000000e+00,0.000000,0.000000e+00,2.333278e-04,0.000211,0.0,0.0,0.000000,0.000000,0.000407,0.000016,0.000043,0
18400,blinkit_D112_718317,2023-06-30,7.300000,4.766667,3.424661,4.858333,0.000283,1.849831e-04,1.329030e-04,1.885405e-04,718317,D112,blinkit,H&C,388.076436,0.000171,0.0,0.000000,4.4,0.000171,2026-07-31,2.480278,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,4.691112,4.003012,4.4,7.300000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,82.500000,344.444444,0.000000,0.0,7.300000,0.000283,0.000000e+00,0.000000,0.000000e+00,1.8205

In [129]:
final_df['platform_name'].unique()

array(['blinkit', 'swiggy', 'zepto'], dtype=object)

In [130]:
trend_df = trend_df.merge(threshold_df, on = ['platform_name', 'depot','parent_material_code', 'run_month'], how = 'left')
trend_df

,platform_name,depot,parent_material_code,run_month,trend_flag,p3m_slope_flag,final_trend,lower_threshold,upper_threshold,mean_value,std_value
0,blinkit,D112,718288,2026-08-31,0,0,0,0.000000,0.000000,0.000000,0.000000
1,blinkit,D112,718310,2026-08-31,-1,0,0,0.000000,0.003047,0.000397,0.000883
2,blinkit,D112,718312,2026-08-31,1,1,1,0.004933,0.063175,0.028230,0.011648
3,blinkit,D112,718317,2026-08-31,-1,0,0,0.000000,0.000000,0.000000,0.000000
4,blinkit,D112,718321,2026-08-31,-1,0,0,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
10999,zepto,D677,810372,2026-08-31,0,0,0,0.000000,0.000000,0.000000,0.000000
11000,zepto,D677,810405,2026-08-31,0,0,0,0.000000,0.000000,0.000000,0.000000
11001,zepto,D677,810406,2026-08-31,-1,0,0,0.000000,0.000000,0.000000,0.000000
11002,zepto,D677,810407,2026-08-31,0,0,0,0.000000,0.000000,0.000000,0.000000


In [131]:
# trend_df.to_csv('t_thres_df_qcom.csv')

In [132]:
final_df = final_df.merge(seasonality_df, on = ["platform_name", "brand_code", 'run_month'], how = 'left')
final_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,depot,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,seasonality_flag
0,blinkit_D112_718589,2024-05-31,0.5,0.25,0.627385,1.151,0.000025,0.000012,0.000031,0.000057,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000075,0.0,0.0,1.5,0.000075,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,0.899318,0.791007,1.5,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000045,0.000039,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0,0
1,blinkit_D112_718589,2024-06-30,0.5,0.25,0.000000,1.061,0.000025,0.000012,0.000000,0.000053,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,0.0,0.0,0.0,0.000000,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,0.108613,0.000000,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000005,0.000000,0.0,0.0,0.0,0.0,0.000075,0.000000,0.000000,0,0
2,blinkit_D112_718589,2024-07-31,0.5,0.25,0.383315,0.145,0.000025,0.000012,0.000019,0.000007,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,0.0,0.0,0.0,0.000000,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,0.611016,0.498236,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000030,0.000025,0.0,0.0,0.0,0.0,0.000000,0.000075,0.000000,0,0
3,blinkit_D112_718589,2024-08-31,0.5,0.25,0.482854,0.250,0.000025,0.000012,0.000024,0.000012,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,0.0,0.0,0.0,0.000000,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,0.713960,0.596314,0.0,0.50,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.50,0.000025,0.0,0.0,0.0,0.000035,0.000030,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000075,0,0
4,blinkit_D112_718589,2024-09-30,0.0,0.25,0.085282,0.351,0.000000,0.000012,0.000004,0.000017,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,0.0,0.0,0.0,0.000000,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,0.311412,0.178123,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,-100.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000015,0.000009,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
317454,zepto_D674_810125,2026-11-30,0.0,0.00,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,810125,D674,zepto,SW_SGPRF,1712.605337,0.000000,0.0,0.0,0.0,0.000000,NaT,0.000000,NaN,NaN,NaN,2026-08-31,M+3,Male Grooming,0.000000,0.000000,0.0,0.76,0.0,0.0,0.0,0.0,0.0,0.0,280.0,inf,0.0,0.0,0.76,0.000130,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000192,0.000069,0.000000,1,0
317455,zepto_D674_810125,2026-12-31,0.0,0.00,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,810125,D674,zepto,SW_SGPRF,1712.605337,0.000000,0.0,0.0,0.0,0.000000,NaT,0.000000,NaN,NaN,NaN,2026-08-31,M+4,Male Grooming,0.000000,0.000000,0.0,0.76,0.0,0.0,0.0,0.0,0.0,0.0,280.0,inf,0.0,0.0,0.76,0.000130,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000192,0.000069,0.000000,1,0
317456,zepto_D674_810125,2027-01-31,0.0,0.00,0.0000

In [133]:
trend_df.columns

Index(['platform_name', 'depot', 'parent_material_code', 'run_month',
       'trend_flag', 'p3m_slope_flag', 'final_trend', 'lower_threshold',
       'upper_threshold', 'mean_value', 'std_value'],
      dtype='object')

In [134]:
final_df = final_df.merge(trend_df[['platform_name', 'depot','parent_material_code', 'run_month',
                                    'final_trend','lower_threshold', 'upper_threshold']], on = ["platform_name", "depot","parent_material_code", 'run_month'], how = 'left')
final_df


,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,depot,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,seasonality_flag,final_trend,lower_threshold,upper_threshold
0,blinkit_D112_718589,2024-05-31,0.5,0.25,0.627385,1.151,0.000025,0.000012,0.000031,0.000057,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000075,0.0,0.0,1.5,0.000075,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,0.899318,0.791007,1.5,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000045,0.000039,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0,0,1,0.000019,0.000251
1,blinkit_D112_718589,2024-06-30,0.5,0.25,0.000000,1.061,0.000025,0.000012,0.000000,0.000053,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,0.0,0.0,0.0,0.000000,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,0.108613,0.000000,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000005,0.000000,0.0,0.0,0.0,0.0,0.000075,0.000000,0.000000,0,0,1,0.000019,0.000251
2,blinkit_D112_718589,2024-07-31,0.5,0.25,0.383315,0.145,0.000025,0.000012,0.000019,0.000007,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,0.0,0.0,0.0,0.000000,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,0.611016,0.498236,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000030,0.000025,0.0,0.0,0.0,0.0,0.000000,0.000075,0.000000,0,0,1,0.000019,0.000251
3,blinkit_D112_718589,2024-08-31,0.5,0.25,0.482854,0.250,0.000025,0.000012,0.000024,0.000012,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,0.0,0.0,0.0,0.000000,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,0.713960,0.596314,0.0,0.50,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.50,0.000025,0.0,0.0,0.0,0.000035,0.000030,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000075,0,0,1,0.000019,0.000251
4,blinkit_D112_718589,2024-09-30,0.0,0.25,0.085282,0.351,0.000000,0.000012,0.000004,0.000017,718589,D112,blinkit,ADV-AHO-R,496.828458,0.000000,0.0,0.0,0.0,0.000000,2026-07-31,1.123439,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Hair Oils,0.311412,0.178123,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,-100.0,0.0,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,0.000015,0.000009,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0,0,1,0.000019,0.000251
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
317454,zepto_D674_810125,2026-11-30,0.0,0.00,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,810125,D674,zepto,SW_SGPRF,1712.605337,0.000000,0.0,0.0,0.0,0.000000,NaT,0.000000,NaN,NaN,NaN,2026-08-31,M+3,Male Grooming,0.000000,0.000000,0.0,0.76,0.0,0.0,0.0,0.0,0.0,0.0,280.0,inf,0.0,0.0,0.76,0.000130,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000192,0.000069,0.000000,1,0,0,0.000065,0.000260
317455,zepto_D674_810125,2026-12-31,0.0,0.00,0.000000,0.000,0.000000,0.000000,0.000000,0.000000,810125,D674,zepto,SW_SGPRF,1712.605337,0.000000,0.0,0.0,0.0,0.000000,NaT,0.000000,NaN,NaN,NaN,2026-08-31,M+4,Male Grooming,0.000000,0.000000,0.0,0.76,0.0,0.0,

In [135]:
# final_df[final_df['skipped'] == 1]['lower_threshold'].sum()

In [136]:
final_df = final_df.sort_values(['key', 'run_month','month_date'])

# base LY


# LY lags
final_df['ly_lag1_value'] = (
    final_df
    .groupby(['key', 'run_month'])['vol_in_rum_value']
    .shift(13)
)

final_df['ly_lag2_value'] = (
    final_df
    .groupby(['key', 'run_month'])['vol_in_rum_value']
    .shift(14)
)

# LY leads
final_df['ly_lead1_value'] = (
    final_df
    .groupby(['key', 'run_month'])['vol_in_rum_value']
    .shift(11)
)

final_df['ly_lead2_value'] = (
    final_df
    .groupby(['key', 'run_month'])['vol_in_rum_value']
    .shift(10)
)


In [137]:
# final_df['stat_bias'] = final_df['Stat Error']/final_df['Actuals Val']
# final_df = final_df.fillna(0)

# import numpy as np
# import pandas as pd

# final_df['stat_bias'] = (
#     final_df['stat_bias']
#     .replace([np.inf, -np.inf], 0)
#     .fillna(0)
# )

# bins = [-np.inf, -0.15, -0.10, -0.05, 0, 0.05, 0.10, 0.15, np.inf]
# labels = [
#     '< -15%',
#     '-15% to -10%',
#     '-10% to -5%',
#     '-5% to 0%',
#     '0% to 5%',
#     '5% to 10%',
#     '10% to 15%',
#     '> 15%'
# ]

# final_df['stat_bias_bucket'] = pd.cut(
#     final_df['stat_bias'],
#     bins=bins,
#     labels=labels,
#     right=False  
# )


In [138]:
# final_df[(final_df['month_date'] == '2026-03-31') & (final_df['run_month'] == '2026-01-31')]['P3M_value'].sum()

In [139]:
brand_seas = pd.read_excel('/data/aman_singh/acuuracy_check/seasonality.xlsx', sheet_name = 'brand')
brand_seas.columns = brand_seas.columns.str.lower()
brand_seas.rename(columns={'brand':'brand_code', 'months_num':'month', 'flag':'is_seasonal_month'}, inplace=True)

final_df['month_date'] = pd.to_datetime(final_df['month_date'])
final_df['month'] = final_df['month_date'].dt.month
final_df = final_df.merge(brand_seas, on = ['brand_code', 'month'], how = 'left')
final_df['is_seasonal_month'].fillna(0, inplace=True)

psku_seas = pd.read_excel('/data/aman_singh/acuuracy_check/seasonality.xlsx', sheet_name = 'psku')
psku_seas.columns = psku_seas.columns.str.lower()
psku_seas.rename(columns={'months_num':'month', 'flag':'is_seasonal_month_psku'}, inplace=True)

final_df = final_df.merge(psku_seas[['parent_material_code', 'month','is_seasonal_month_psku']], on = ['parent_material_code', 'month'], how = 'left')
final_df['is_seasonal_month_psku'].fillna(0, inplace=True)
final_df['final_seasonal_month'] = np.where(
    (final_df['is_seasonal_month'] == 1) | (final_df['is_seasonal_month_psku'] == 1), 1, 0
)



In [140]:
final_df['run_month'] = pd.to_datetime(final_df['run_month'])
def compute_adjusted_pm(df_grp, window, column, year_shift=0):
    df_grp = df_grp.sort_values("month_date")

    run_month = df_grp["run_month"].iloc[0]

    # define cutoff
    end_date = run_month - pd.DateOffset(years=year_shift)

    # keep only eligible history (before run month & non-event)
    hist = df_grp[
        (df_grp["month_date"] < end_date) &
        (df_grp["final_seasonal_month"] == 0)
    ]

    if hist.empty:
        return np.nan

    # take last `window` non-event months
    hist = hist.tail(window)

    # if len(hist) < window:
    #     return np.nan   # optional, keeps behavior strict

    return hist[column].mean()

adj_df = final_df.groupby(
    ['key', "run_month"]
).apply(
    lambda x: pd.Series({
        "P3M_non_seasonal": compute_adjusted_pm(x, 3,'vol_in_rum',0),
        "P6M_non_seasonal": compute_adjusted_pm(x, 6,'vol_in_rum',0),
        "P3M_non_seasonal_value": compute_adjusted_pm(x, 3,'vol_in_rum_value',0),
        "P6M_non_seasonal_value": compute_adjusted_pm(x, 6,'vol_in_rum_value',0)
    })
).reset_index()
adj_df


adj_ly_df = final_df.groupby(
    ['key', "run_month"]
).apply(
    lambda x: pd.Series({
        "LY_P3M_non_seasonal": compute_adjusted_pm(x, 3,'vol_in_rum',year_shift=1),
        "LY_P6M_non_seasonal": compute_adjusted_pm(x, 6,'vol_in_rum', year_shift=1),
        "LY_P3M_non_seasonal_value": compute_adjusted_pm(x, 3,'vol_in_rum_value', year_shift=1),
        "LY_P6M_non_seasonal_value": compute_adjusted_pm(x, 6,"vol_in_rum_value", year_shift=1)
    })
).reset_index()

adj_df = adj_df.merge(adj_ly_df, on = ['key', 'run_month'], how = 'left')
#adj_df[adj_df['key'] == 'reliance_b2c_2_haryana_718488']
adj_df

,key,run_month,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value
0,blinkit_D112_718288,2026-08-31,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000e+00
1,blinkit_D112_718310,2026-08-31,0.000000,0.000000,0.0000,0.000000,0.074167,0.058875,0.002590,2.056351e-03
2,blinkit_D112_718312,2026-08-31,1.331333,1.016167,0.0465,0.035492,0.851333,0.874500,0.029735,3.054401e-02
3,blinkit_D112_718317,2026-08-31,0.000000,0.000000,0.0000,0.000000,0.000000,0.016667,0.000000,6.467941e-07
4,blinkit_D112_718321,2026-08-31,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...
10999,zepto_D677_810372,2026-08-31,0.000000,0.000000,0.0000,0.000000,0.000000,0.025000,0.000000,4.448184e-06
11000,zepto_D677_810405,2026-08-31,0.000000,0.000000,0.0000,0.000000,0.000000,0.010000,0.000000,1.779274e-06
11001,zepto_D677_810406,2026-08-31,0.000000,0.000000,0.0000,0.000000,0.000000,0.005000,0.000000,8.896369e-07
11002,zepto_D677_810407,2026-08-31,0.000000,0.000000,0.0000,0.000000,0.000000,0.005000,0.000000,8.896369e-07


In [141]:
final_df.shape

(317459, 70)

In [142]:
# adj_df.to_csv('seasonal_p3m_qcom.csv', index=False)
#all[all['month_date'].isin(['2025-11-30','2025-12-31','2026-01-31'])].groupby(['key','run_month','month_date','final_seasonal_month'])['vol_in_rum'].sum().reset_index().to_csv('seasonal_month_check.csv', index=False)
final_df = final_df.merge(
    adj_df,
    on=['key','run_month'],
    how="left"
)
final_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,depot,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,seasonality_flag,final_trend,lower_threshold,upper_threshold,ly_lag1_value,ly_lag2_value,ly_lead1_value,ly_lead2_value,month,is_seasonal_month,seasonal_months,is_seasonal_month_psku,final_seasonal_month,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value
0,blinkit_D112_718288,2024-05-31,0.048333,0.024167,0.054867,0.067667,0.000671,0.000336,0.000762,0.000940,718288,D112,blinkit,SAFF GOLD,138865.260689,0.002014,0.0,0.0,0.145,0.002014,2026-07-31,5.196152,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Saffola Oils,0.065557,0.059909,0.145,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000910,0.000832,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0,0,0,0.0,0.0,NaN,NaN,NaN,NaN,5,0.0,NaN,0.0,0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000
1,blinkit_D112_718288,2024-06-30,0.048333,0.024167,0.016942,0.019333,0.000671,0.000336,0.000235,0.000268,718288,D112,blinkit,SAFF GOLD,138865.260689,0.000000,0.0,0.0,0.000,0.000000,2026-07-31,5.196152,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Saffola Oils,0.027916,0.022502,0.000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000388,0.000312,0.0,0.0,0.0,0.0,0.002014,0.000000,0.000000,0,0,0,0.0,0.0,NaN,NaN,NaN,NaN,6,0.0,NaN,0.0,0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000
2,blinkit_D112_718288,2024-07-31,0.048333,0.024167,0.011667,0.004833,0.000671,0.000336,0.000162,0.000067,718288,D112,blinkit,SAFF GOLD,138865.260689,0.000000,0.0,0.0,0.000,0.000000,2026-07-31,5.196152,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Saffola Oils,0.021299,0.015508,0.000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000296,0.000215,0.0,0.0,0.0,0.0,0.000000,0.002014,0.000000,0,0,0,0.0,0.0,NaN,NaN,NaN,NaN,7,0.0,NaN,0.0,0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000
3,blinkit_D112_718288,2024-08-31,0.048333,0.024167,0.011857,0.009667,0.000671,0.000336,0.000165,0.000134,718288,D112,blinkit,SAFF GOLD,138865.260689,0.000000,0.0,0.0,0.000,0.000000,2026-07-31,5.196152,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Saffola Oils,0.023329,0.018829,0.000,0.048333,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.048333,0.000671,0.0,0.000000,0.000000,0.000324,0.000261,0.0,0.0,0.0,0.0,0.000000,0.000000,0.002014,0,0,0,0.0,0.0,NaN,NaN,NaN,NaN,8,0.0,NaN,0.0,0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000
4,blinkit_D112_718288,2024-09-30,0.000000,0.024167,0.004444,0.000000,0.000000,0.000336,0.000062,0.000000,718288,D112,blinkit,SAFF GOLD,138865.260689,0.000000,0.0,0.0,0.000,0.000000,2026-07-31,5.196152,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Saffola Oils,0.017095,0.010392,0.000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,-100.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000237,0.000144,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0,0,0,0.0,0.0,NaN,NaN,NaN,NaN,9,0.0,NaN,

In [143]:
final_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,depot,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,seasonality_flag,final_trend,lower_threshold,upper_threshold,ly_lag1_value,ly_lag2_value,ly_lead1_value,ly_lead2_value,month,is_seasonal_month,seasonal_months,is_seasonal_month_psku,final_seasonal_month,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value
0,blinkit_D112_718288,2024-05-31,0.048333,0.024167,0.054867,0.067667,0.000671,0.000336,0.000762,0.000940,718288,D112,blinkit,SAFF GOLD,138865.260689,0.002014,0.0,0.0,0.145,0.002014,2026-07-31,5.196152,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Saffola Oils,0.065557,0.059909,0.145,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000910,0.000832,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0,0,0,0.0,0.0,NaN,NaN,NaN,NaN,5,0.0,NaN,0.0,0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000
1,blinkit_D112_718288,2024-06-30,0.048333,0.024167,0.016942,0.019333,0.000671,0.000336,0.000235,0.000268,718288,D112,blinkit,SAFF GOLD,138865.260689,0.000000,0.0,0.0,0.000,0.000000,2026-07-31,5.196152,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Saffola Oils,0.027916,0.022502,0.000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000388,0.000312,0.0,0.0,0.0,0.0,0.002014,0.000000,0.000000,0,0,0,0.0,0.0,NaN,NaN,NaN,NaN,6,0.0,NaN,0.0,0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000
2,blinkit_D112_718288,2024-07-31,0.048333,0.024167,0.011667,0.004833,0.000671,0.000336,0.000162,0.000067,718288,D112,blinkit,SAFF GOLD,138865.260689,0.000000,0.0,0.0,0.000,0.000000,2026-07-31,5.196152,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Saffola Oils,0.021299,0.015508,0.000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000296,0.000215,0.0,0.0,0.0,0.0,0.000000,0.002014,0.000000,0,0,0,0.0,0.0,NaN,NaN,NaN,NaN,7,0.0,NaN,0.0,0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000
3,blinkit_D112_718288,2024-08-31,0.048333,0.024167,0.011857,0.009667,0.000671,0.000336,0.000165,0.000134,718288,D112,blinkit,SAFF GOLD,138865.260689,0.000000,0.0,0.0,0.000,0.000000,2026-07-31,5.196152,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Saffola Oils,0.023329,0.018829,0.000,0.048333,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.048333,0.000671,0.0,0.000000,0.000000,0.000324,0.000261,0.0,0.0,0.0,0.0,0.000000,0.000000,0.002014,0,0,0,0.0,0.0,NaN,NaN,NaN,NaN,8,0.0,NaN,0.0,0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000
4,blinkit_D112_718288,2024-09-30,0.000000,0.024167,0.004444,0.000000,0.000000,0.000336,0.000062,0.000000,718288,D112,blinkit,SAFF GOLD,138865.260689,0.000000,0.0,0.0,0.000,0.000000,2026-07-31,5.196152,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Saffola Oils,0.017095,0.010392,0.000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,-100.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000237,0.000144,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0,0,0,0.0,0.0,NaN,NaN,NaN,NaN,9,0.0,NaN,

In [144]:
all_brand = final_df.groupby(['brand_code', 'run_month','month_date'])['vol_in_rum_value'].sum().reset_index()
def detect_month_anomaly(df, brand_code, month_num, threshold=0.25, months_window=3):
    """
    Detect if a specific month's vol_in_rum_value is >25% different 
    from past 3 months & next 3 months, and if pattern repeats in last 2 years.
    
    Parameters:
    - df: input dataframe with 'month_date', 'vol_in_rum_value'
    - brand_code: filter by this brand code
    - month_num: month to check (6, 7, 8, 9)
    - threshold: 25% difference threshold
    - months_window: number of months before and after to compare
    """
    
    df_brand = df[df['brand_code'] == brand_code].sort_values('month_date').copy()
    
    if df_brand.empty:
        return None
    
    df_brand['year'] = df_brand['month_date'].dt.year
    df_brand['month'] = df_brand['month_date'].dt.month
    
    years = sorted(df_brand['year'].unique())
    current_year = years[-1]
    past_years = [y for y in years if y < current_year][-2:]
    
    anomalies = []
    
    for year in past_years:
        df_year = df_brand[df_brand['year'] == year].sort_values('month_date')
        
        month_data = df_year[df_year['month'] == month_num]
        if month_data.empty:
            continue
        
        month_value = month_data['vol_in_rum_value'].iloc[0]
        
        past_months = [(month_num - i - 1) % 12 for i in range(1, months_window + 1)]
        next_months = [(month_num + i - 1) % 12 + 1 for i in range(1, months_window + 1)]
        
        past_m = df_year[df_year['month'].isin(past_months)]['vol_in_rum_value']
        next_m = df_year[df_year['month'].isin(next_months)]['vol_in_rum_value']
        
        comparison_values = pd.concat([past_m])
        
        if comparison_values.empty:
            continue
        
        pct_diffs = []
        for comp_value in comparison_values:
            if comp_value != 0:
                pct_diff = (month_value - comp_value) / comp_value
                pct_diffs.append(pct_diff)
        
        if pct_diffs:
            positive_diffs = [p for p in pct_diffs if p > 0]
            negative_diffs = [p for p in pct_diffs if p < 0]
            same_sign = len(positive_diffs) == len(pct_diffs) or len(negative_diffs) == len(pct_diffs)
            is_anomaly = len([p for p in pct_diffs if abs(p) > threshold]) == len(pct_diffs) and same_sign
        else:
            is_anomaly = False

        anomalies.append({
            'brand_code': brand_code,
            'month': month_num,
            'year': year,
            'month_value': month_value,
            'num_months_compared': len(comparison_values),
            'pct_diffs_from_each': pct_diffs,
            'min_pct_diff': min(pct_diffs) * 100 if pct_diffs else None,
            'max_pct_diff': max(pct_diffs) * 100 if pct_diffs else None,
            'is_anomaly': is_anomaly,
            'direction': 'higher' if month_value > comparison_values.mean() else 'lower'
        })
    
    if len(anomalies) == 2:
        pattern_repeats = anomalies[0]['is_anomaly'] and anomalies[1]['is_anomaly']
        return pd.DataFrame(anomalies), pattern_repeats
    
    return pd.DataFrame(anomalies), False


# Check months 6, 7, 8, 9
brands = all_brand['brand_code'].unique()
results = []

for month in [9, 10,11,12]:
    for brand in brands:
        df_result, repeats = detect_month_anomaly(all_brand, brand, month)
        if df_result is not None and not df_result.empty:
            df_result['pattern_repeats'] = repeats
            results.append(df_result)

anomaly_summary = pd.concat(results, ignore_index=True)
print(anomaly_summary[anomaly_summary['pattern_repeats'] == True])

    brand_code  month  year  month_value  num_months_compared  \
18     KAYA_GM      9  2025     0.021946                    3   
19     KAYA_GM      9  2026     0.000000                    3   
20     KAYA_ML      9  2025     0.021332                    3   
21     KAYA_ML      9  2026     0.000000                    3   
65   PADV_SWGL      9  2024     0.010468                    3   
..         ...    ...   ...          ...                  ...   
558  SF_SOYBRJ     11  2025     0.000000                    3   
561   SW HSPRY     11  2025     0.129586                    3   
562   SW HSPRY     11  2026     0.000000                    3   
713   SAF_MAYO     12  2024     0.001356                    3   
714   SAF_MAYO     12  2025     0.000000                    3   

                                   pct_diffs_from_each  min_pct_diff  \
18   [-0.46891607182164086, -0.6978441721084175, -0...    -69.784417   
19                                              [-1.0]   -100.000000   
20 

In [145]:
# mnth = 6
# all_brand = final_df.groupby(['brand_code', 'run_month_x','month_date'])['vol_in_rum_value'].sum().reset_index()
# def detect_april_anomaly(df, brand_code, threshold=0.25, months_window=3):
#     """
#     Detect if April's vol_in_rum_value is >25% different 
#     from past 3 months & next 3 months, and if pattern repeats in last 2 years.
    
#     Parameters:
#     - df: input dataframe with 'month_date', 'vol_in_rum_value', 'run_month'
#     - brand_code: filter by this brand code
#     - threshold: 25% difference threshold
#     - months_window: number of months before and after April to compare
    
#     """
    
#     # Filter for brand and sort by month_date
#     df_brand = df[df['brand_code'] == brand_code].sort_values('month_date').copy()
    
#     if df_brand.empty:
#         return None
    
#     # Extract year and month
#     df_brand['year'] = df_brand['month_date'].dt.year
#     df_brand['month'] = df_brand['month_date'].dt.month
    
#     # Get unique years (excluding current year if incomplete)
#     years = sorted(df_brand['year'].unique())
#     current_year = years[-1]
#     past_years = [y for y in years if y < current_year][-2:]  # Last 2 years
    
#     anomalies = []
    
#     # Check each past year's April
#     for year in past_years:
#         df_year = df_brand[df_brand['year'] == year].sort_values('month_date')
        
#         # Get April data (month == 4)
#         april_data = df_year[df_year['month'] == mnth]
#         if april_data.empty:
#             continue
        
#         april_value = april_data['vol_in_rum_value'].iloc[0]
#         april_month = mnth
        
#         # Dynamically calculate past and next months
#         past_months = [(april_month - i - 1) % 12 + 1 for i in range(1,months_window+1)]
#         print(past_months)
#         next_months = [(april_month + i - 1) % 12 + 1 for i in range(1, months_window + 1)]
#         print(next_months)
        
#         # Get past and next months values
#         past_3m = df_year[df_year['month'].isin(past_months)]['vol_in_rum_value']
#         next_3m = df_year[df_year['month'].isin(next_months)]['vol_in_rum_value']
        
#         # Combine all comparison months
#         comparison_values = pd.concat([past_3m, next_3m])
        
#         if comparison_values.empty:
#             continue
        
#         # Calculate mean of comparison months
#         #mean_value = comparison_values.mean()
        
#         # Calculate percentage difference
#         pct_diffs = []
#         for comp_value in comparison_values:
#             if comp_value != 0:
#                 pct_diff = (april_value - comp_value) / comp_value
#                 pct_diffs.append(pct_diff)
        
#         # April is anomalous if it's >25% different from ALL comparison months
#         # AND all differences have the same sign (all positive or all negative)
#         if pct_diffs:
#             positive_diffs = [p for p in pct_diffs if p > 0]
#             negative_diffs = [p for p in pct_diffs if p < 0]
#             same_sign = len(positive_diffs) == len(pct_diffs) or len(negative_diffs) == len(pct_diffs)
#             is_anomaly = len([p for p in pct_diffs if abs(p) > threshold]) == len(pct_diffs) and same_sign
#         else:
#             is_anomaly = False

#         anomalies.append({
#             'brand_code': brand_code,
#             'year': year,
#             'april_value': april_value,
#             'num_months_compared': len(comparison_values),
#             'pct_diffs_from_each': pct_diffs,
#             'min_pct_diff': min(pct_diffs) * 100 if pct_diffs else None,
#             'max_pct_diff': max(pct_diffs) * 100 if pct_diffs else None,
#             'is_anomaly': is_anomaly,
#             'direction': 'higher' if april_value > comparison_values.mean() else 'lower'
#         })
    
#     # Check if pattern repeats in both years
#     if len(anomalies) == 2:
#         pattern_repeats = anomalies[0]['is_anomaly'] and anomalies[1]['is_anomaly']
#         return pd.DataFrame(anomalies), pattern_repeats
    
#     return pd.DataFrame(anomalies), False


# # Usage: Apply to each brand code
# brands = all_brand['brand_code'].unique()
# results = []

# for brand in brands:
#     df_result, repeats = detect_april_anomaly(all_brand, brand)
#     if df_result is not None and not df_result.empty:
#         df_result['pattern_repeats'] = repeats
#         results.append(df_result)


# anomaly_summary = pd.concat(results, ignore_index=True)
# print(anomaly_summary[anomaly_summary['pattern_repeats'] == True])

In [146]:
final_df.shape

(317459, 78)

In [147]:
final_brands = anomaly_summary[anomaly_summary['pattern_repeats'] == True].drop_duplicates(subset=['brand_code','month'])[['brand_code', 'month','direction', 'min_pct_diff', 'max_pct_diff']]
final_brands['month_different'] = 1
final_df['month'] = final_df['month_date'].dt.month
final_df = final_df.merge(final_brands[['brand_code', 'month','month_different']], on = ['brand_code','month'], how = 'left')
final_df['month_different'].fillna(0, inplace=True)
final_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,depot,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,seasonality_flag,final_trend,lower_threshold,upper_threshold,ly_lag1_value,ly_lag2_value,ly_lead1_value,ly_lead2_value,month,is_seasonal_month,seasonal_months,is_seasonal_month_psku,final_seasonal_month,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value,month_different
0,blinkit_D112_718288,2024-05-31,0.048333,0.024167,0.054867,0.067667,0.000671,0.000336,0.000762,0.000940,718288,D112,blinkit,SAFF GOLD,138865.260689,0.002014,0.0,0.0,0.145,0.002014,2026-07-31,5.196152,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Saffola Oils,0.065557,0.059909,0.145,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000910,0.000832,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0,0,0,0.0,0.0,NaN,NaN,NaN,NaN,5,0.0,NaN,0.0,0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0
1,blinkit_D112_718288,2024-06-30,0.048333,0.024167,0.016942,0.019333,0.000671,0.000336,0.000235,0.000268,718288,D112,blinkit,SAFF GOLD,138865.260689,0.000000,0.0,0.0,0.000,0.000000,2026-07-31,5.196152,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Saffola Oils,0.027916,0.022502,0.000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000388,0.000312,0.0,0.0,0.0,0.0,0.002014,0.000000,0.000000,0,0,0,0.0,0.0,NaN,NaN,NaN,NaN,6,0.0,NaN,0.0,0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0
2,blinkit_D112_718288,2024-07-31,0.048333,0.024167,0.011667,0.004833,0.000671,0.000336,0.000162,0.000067,718288,D112,blinkit,SAFF GOLD,138865.260689,0.000000,0.0,0.0,0.000,0.000000,2026-07-31,5.196152,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Saffola Oils,0.021299,0.015508,0.000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000296,0.000215,0.0,0.0,0.0,0.0,0.000000,0.002014,0.000000,0,0,0,0.0,0.0,NaN,NaN,NaN,NaN,7,0.0,NaN,0.0,0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0
3,blinkit_D112_718288,2024-08-31,0.048333,0.024167,0.011857,0.009667,0.000671,0.000336,0.000165,0.000134,718288,D112,blinkit,SAFF GOLD,138865.260689,0.000000,0.0,0.0,0.000,0.000000,2026-07-31,5.196152,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Saffola Oils,0.023329,0.018829,0.000,0.048333,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.048333,0.000671,0.0,0.000000,0.000000,0.000324,0.000261,0.0,0.0,0.0,0.0,0.000000,0.000000,0.002014,0,0,0,0.0,0.0,NaN,NaN,NaN,NaN,8,0.0,NaN,0.0,0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0
4,blinkit_D112_718288,2024-09-30,0.000000,0.024167,0.004444,0.000000,0.000000,0.000336,0.000062,0.000000,718288,D112,blinkit,SAFF GOLD,138865.260689,0.000000,0.0,0.0,0.000,0.000000,2026-07-31,5.196152,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-08-31,None,Saffola Oils,0.017095,0.010392,0.000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,-100.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000237,0.000144,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0,0,0,0.

In [148]:
final_df['run_month'].unique()

<DatetimeArray>
['2026-08-31 00:00:00']
Length: 1, dtype: datetime64[ns]

In [151]:
final_df[(final_df['run_month'] == '2026-08-31')&(final_df['month_date'] == '2026-09-30')]['P3M_value'].sum()

31.51346882325608

In [152]:
missing_df[(missing_df['run_month'] == '2026-08-31')&(missing_df['month_date'] == '2026-09-30') ]['P3M_value'].sum()

1.429657997026749

In [ ]:
final_df[(final_df['run_month_x'] == '2026-04-30')&(final_df['month_date'] == '2026-05-31') & (final_df['key'] == 'blinkit_D112_718288')]

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,parent_material_code,depot,platform_name,brand_code,qtr_ind_rate,vol_in_rum_value,pred_best_model,pred_value_best_model,vol_in_rum_treated,vol_in_rum_value_treated,train_till,cov,run,step,file_path,run_month_x,M month,portfolio,pred_prophet_70%ile,pred_prophet_60%ile,vol_in_rum,P3M,P6M,LY P3M,LY P6M,LY P3M_copy,P3M Max,P3M Top 2 Mean,MoM P3M growth,MoM P3M growth_lag_1,MoM P3M growth_lag_2,>=20%_3M_inc_month_count,"Avg(P3M Mean, Max)",P3M_value,P6M_value,LY P3M_value,LY P6M_value,pred_prophet_70%ile_value,pred_prophet_60%ile_value,LY,LLY,LY value,LLY value,OT_Value_in_Cr_lag_1,OT_Value_in_Cr_lag_2,OT_Value_in_Cr_lag_3,skipped,seasonality_flag,final_trend,lower_threshold,upper_threshold,ly_lag1_value,ly_lag2_value,ly_lead1_value,ly_lead2_value,month,is_seasonal_month,seasonal_months,is_seasonal_month_psku,final_seasonal_month,run_month_y,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value,month_different
102,blinkit_D112_718288,2026-05-31,0.0,0.0,0.037482,0.012221,0.0,0.0,0.00052,0.00017,718288,D112,blinkit,SAFF GOLD,138865.260689,0.0,0.037482,0.00052,0.0,0.0,2026-03-31,4.795832,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-04-30,M+1,Saffola Oils,0.049023,0.043575,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-100.0,-100.0,-100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000681,0.000605,0.0,0.145,0.0,0.002014,0.0,0.0,0.0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,5,0.0,NaN,0.0,0,2026-03-31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
103,blinkit_D112_718288,2026-05-31,0.0,0.0,0.037482,0.012221,0.0,0.0,0.00052,0.00017,718288,D112,blinkit,SAFF GOLD,138865.260689,0.0,0.037482,0.00052,0.0,0.0,2026-03-31,4.795832,run,acuuracy_check,/data/aman_singh/acuuracy_check/trend_file_tra...,2026-04-30,M+1,Saffola Oils,0.049023,0.043575,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-100.0,-100.0,-100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000681,0.000605,0.0,0.145,0.0,0.002014,0.0,0.0,0.0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,5,0.0,NaN,0.0,0,2026-04-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [153]:
final_df.to_csv('/data/aman_singh/acuuracy_check/all_combination_qcom_trend.csv')

In [155]:
final_df[(final_df['M month'].notna())].to_csv('/data/aman_singh/acuuracy_check/all_combination_qcom_aug_pred.csv')